# pMRI-iUS Slice-to-Volume 3DOF Estimation Network
## By: Olive Schonfeldt (SCHOLI016)
## July - October 2025

The main objective of this algorithm is to determine the exact position and orientation of a 2D intra-operative ultrasound (iUS) slice with respect to a pre-operative 3D magnetic resonance imaging (pMRI) volume. It does this by predicting three parameters, known as 3 Degrees of Freedom (3DOF):

- Z-Translation: The slice's depth or "height" within the volume.
- X-Rotation: The rotation around the X-axis (roll).
- Y-Rotation: The rotation around the Y-axis (pitch).

The algorithm tackles this complex problem by breaking it into four sequential stages, starting with a coupled estimate and progressively refining it.
### Stage 1: Coupled Regression
- Predicts an estimate for Z-translation, X-rotation, and Y-rotation simultaneously.
- The output for each parameter is a Gaussian distribution (mean and sigma), capturing the prediction and its uncertainty.

### Stage 2: Z-Translation Refinement

- Takes the Z prediction from Stage 1 as a conceptual starting point.
- It performs a classification task to pinpoint the exact Z-slice within the MRI volume.

### Stage 3: X-Rotation Refinement

- Takes the original US slice and the now-corrected MRI slice (using the refined Z-position from Stage 2).
- Performs a classification task to predict the final X-rotation.
- Adopts a limited search window dictated from Stage 1.

### Stage 4: Y-Rotation Refinement

- Similar to Stage 3, it uses the US slice and the corrected MRI slice.
- Performs a classification task with a window to predict the final Y-rotation.




In [ ]:
pip install --upgrade nibabel kornia

# --- 0. Imports ---

In [ ]:
# --- 0. Imports ---

# Import necessary libraries.
# PyTorch for building and training the neural network.
# NumPy for numerical operations.
# Matplotlib & Seaborn for plotting results.
# tqdm for progress bars.
# SciPy for image rotation.
# Other utilities for data handling and metrics.

# --- Standard Library Imports ---
import os
import time
import itertools
from tqdm import tqdm
import warnings

# --- Core Deep Learning Imports (PyTorch) ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision.models as models
from torch.nn import GaussianNLLLoss, CrossEntropyLoss
import torch.nn.functional as F

# --- Numerical and Scientific Computing Imports ---
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate as scipy_rotate
import kornia.geometry.transform as K
import math

# --- Plotting and Visualization Imports ---
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Suppress specific warnings from Kornia that might clutter the output
warnings.filterwarnings("ignore", category=UserWarning, module='kornia')

# --- 1. Data Loading ---

In [ ]:
# --- 1. Data Loading ---

def load_input_data(input_mri_file_path, input_us_file_path):
  """
    Loads a single pair of corresponding MRI and US NIfTI (.nii.gz) files.

    Args:
        input_mri_file_path (str): The full path to the MRI NIfTI file.
        input_us_file_path (str): The full path to the Ultrasound NIfTI file.

    Returns:
        tuple: A tuple containing two NumPy arrays (mri_data, us_data) if successful,
               or (None, None) if files are not found or an error occurs during loading.
    """
  if not os.path.exists(input_mri_file_path) or not os.path.exists(input_us_file_path):
    # Print an error message if one or both files are missing
    print(f"Error: One or both files were not found.")
    print(f"MRI Path: '{input_mri_file_path}'")
    print(f"US Path: '{input_us_file_path}'")
    return None, None

  try:
    # Attempt to load the NIfTI files using nibabel
    mri_img = nib.load(input_mri_file_path)
    us_img = nib.load(input_us_file_path)

    # Extract the image data from the loaded files as NumPy arrays
    mri_data = mri_img.get_fdata()
    us_data = us_img.get_fdata()

    # Return the loaded data arrays
    return mri_data, us_data

  # Catch any exceptions that might occur during file loading or data extraction
  except Exception as e:
    print(f"An error occurred while trying to process the files: {e}")
    return None, None

def load_all_volumes(base_path, case_indices):
  """
    Loads multiple MRI and US volume pairs based on a list of case indices.
    Assumes a consistent file naming convention (e.g., CaseX-FLAIR.nii.gz).

    Args:
        base_path (str): The directory path where the case files are stored.
        case_indices (iterable): A list or range of integer indices representing the cases to load.

    Returns:
        tuple: A tuple containing two lists:
               - mri_vols: A list of loaded MRI volumes (NumPy arrays).
               - us_vols: A list of loaded US volumes (NumPy arrays).
  """
  # Initialize empty lists to store the loaded volumes
  mri_vols = []
  us_vols = []
  print("Loading all MRI and ultrasound volumes...")
  for i in case_indices:
    # Construct the expected file paths based on the index and naming convention
    mri_file = os.path.join(base_path, f"Case{i}-FLAIR.nii.gz")
    us_file = os.path.join(base_path, f"Case{i}-US-during.nii.gz")

    # Display which case is currently being loaded
    print(f"--- Loading Case {i} ---")
    # Call the single-pair loading function
    mri_data, us_data = load_input_data(mri_file, us_file)

    if mri_data is not None and us_data is not None:
      # If successful, append the loaded data to the respective lists
      mri_vols.append(mri_data)
      us_vols.append(us_data)

  # Print a summary of how many pairs were successfully loaded
  print(f"\nSuccessfully loaded {len(mri_vols)} volume pairs.")
  # Return the lists of loaded volumes
  return mri_vols, us_vols

# --- Script Execution: Load Data ---
# Call the function to load all specified case volumes from the "INPUT/" directory.
# This populates the mri_vols and us_vols lists used later in the script.
# (Currently loads cases 2 through 18)
mri_vols, us_vols = load_all_volumes("INPUT/", case_indices=range(2, 19))

# --- 2. Configuration & Search Space ---

In [ ]:
# --- 2. Configuration & Search Space ---

# --- Hardware & Performance Settings ---
# Automatically select GPU ('cuda') if available, otherwise fallback to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = os.cpu_count() // 2 if torch.cuda.is_available() else 0

# --- Training Hyperparameters ---
LEARNING_RATE = 1e-4          # Initial learning rate for the Adam optimizer
BATCH_SIZE = 16               # Number of samples processed in each training iteration
NUM_EPOCHS_S1 = 30            # Number of epochs to train the Stage 1 model
NUM_EPOCHS_REFINEMENT = 20    # Number of epochs to train Stages 2, 3, and 4 (Refinement)

# --- Data & Model Architecture Parameters ---
VOLUME_DIM = 50               # Assumed dimension (Depth, Height, Width) of the input volumes (e.g., 50x50x50)
TRAIN_SPLIT = 0.80            # Proportion of the dataset used for training (80%)
VAL_SPLIT = 0.15              # Proportion of the dataset used for validation (15%)
# Note: TEST_SPLIT is implicitly defined as 1.0 - TRAIN_SPLIT - VAL_SPLIT (5%)
FEATURE_DIM = 256             # Dimensionality of the feature vectors produced by the encoders
NUM_REGRESSION_PARAMS = 3     # Number of parameters predicted by the Stage 1 regression model (z, x_rot, y_rot)

# --- Search Space Definition ---
# Define the discrete set of possible values for each degree of freedom (DOF).
# Used for generating training data and defining classification targets.
# Z-Translations: All possible slice indices from 0 to VOLUME_DIM - 1
Z_TRANSLATIONS = np.arange(VOLUME_DIM)
X_ROTATIONS = np.arange(0, 21, 5)       # Angles from 0 to 20 degrees, in steps of 5 degrees
Y_ROTATIONS = np.arange(0, 21, 5)       # Angles from 0 to 20 degrees, in steps of 5 degrees

# --- Mappings for Classification ---
# Create dictionaries to convert between rotation angles and their corresponding
# integer class indices (0, 1, 2, 3, 4) needed for CrossEntropyLoss.
# Example: x_rot_to_idx[10.0] -> 2
x_rot_to_idx = {float(angle): i for i, angle in enumerate(X_ROTATIONS)}
y_rot_to_idx = {float(angle): i for i, angle in enumerate(Y_ROTATIONS)}
# Create reverse mappings to convert predicted indices back to angles.
# Example: idx_to_x_rot[2] -> 10.0
idx_to_x_rot = {i: float(angle) for angle, i in x_rot_to_idx.items()}
idx_to_y_rot = {i: float(angle) for angle, i in y_rot_to_idx.items()}

# --- Print Configuration Summary ---
# Display key configuration settings to the console for verification.
print(f"--- Configuration ---")
print(f"Using device: {DEVICE}")
print(f"Num Workers for DataLoader: {NUM_WORKERS}")
print(f"Z Translations (Num Classes): {len(Z_TRANSLATIONS)}")
print(f"X Rotations (Num Classes): {len(X_ROTATIONS)}")
print(f"Y Rotations (Num Classes): {len(Y_ROTATIONS)}")
print("-" * 25)

# --- 3. Data Pre-computation & Dataset Classes & CNNs ---

In [ ]:
# --- 3. Data Pre-computation & Dataset Class & CNNs ---

def create_precomputed_dataset(us_vols, mri_vols):
    """
    Generates a large dataset of simulated US slice misalignments offline.

    This function iterates through all possible combinations of Z-translation,
    X-rotation, and Y-rotation defined in the global configuration. For each
    combination, it applies the rotation to the original US volume, extracts the
    corresponding 2D slice, and saves this slice along with the transformation
    parameters (ground truth labels) and the index of the original MRI volume it came from.

    This pre-computation step significantly speeds up training by avoiding
    costly CPU-based rotations within the GPU-accelerated training loop.

    Args:
        us_vols (list): A list of 3D Ultrasound volumes (NumPy arrays).
        mri_vols (list): A list of corresponding 3D MRI volumes (NumPy arrays).

    Returns:
        list: A list of tuples. Each tuple represents one training sample:
              (us_slice_np, mri_vol_idx, labels_dict).
              - us_slice_np: The pre-rotated and sliced 2D US image (NumPy array).
              - mri_vol_idx: The index of the original MRI volume this slice corresponds to.
              - labels_dict: A dictionary containing ground truth labels for all stages
                             ("s1", "s2", "s3", "s4").
    """
    print("Starting dataset pre-computation...")
    # List to store the generated samples
    us_precomputed_data = []
    # Generate all possible combinations of (Z, X_rot, Y_rot) using the defined search spaces
    transform_combinations = list(itertools.product(Z_TRANSLATIONS, X_ROTATIONS, Y_ROTATIONS))

    # Loop through each pair of MRI and US volumes
    for vol_idx, (mri_vol, us_vol) in enumerate(zip(mri_vols, us_vols)):
        # Loop through every possible transformation combination for the current volume pair
        # Use tqdm for a progress bar
        for z_true, x_rot_true, y_rot_true in tqdm(transform_combinations, desc=f"Processing Vol {vol_idx+1}/{len(us_vols)}"):
            # Simulate the misalignment by applying the rotations to the US volume.
            # Apply Y-rotation (pitch) first around axes (0, 2).
            # 'reshape=False' keeps the volume dimensions the same.
            # 'order=1' uses bilinear interpolation.
            # 'mode='constant', cval=0.0' fills outside areas with black.
            rotated_us = scipy_rotate(us_vol, angle=y_rot_true, axes=(0, 2), reshape=False, order=1, mode='constant', cval=0.0)
            # Apply X-rotation (roll) second around axes (0, 1).
            rotated_us = scipy_rotate(rotated_us, angle=x_rot_true, axes=(0, 1), reshape=False, order=1, mode='constant', cval=0.0)

            # Extract the specific 2D slice corresponding to the Z-translation.
            # .copy() ensures it's a separate NumPy array.
            us_slice = rotated_us[int(z_true), :, :].copy()

            # Store the computed slice, the index of the MRI it corresponds to, and labels
            labels = {
                "s1": np.array([z_true, x_rot_true, y_rot_true], dtype=np.float32), # Regression
                "s2": int(z_true),                                                 # Z-Class
                "s3": x_rot_to_idx[float(x_rot_true)],                             # X-Class
                "s4": y_rot_to_idx[float(y_rot_true)],                             # Y-Class
            }
            us_precomputed_data.append((us_slice, vol_idx, labels))

    print(f"Pre-computation finished. Total samples created: {len(us_precomputed_data)}")

    # Return the complete list of pre-computed samples
    return us_precomputed_data

class PrecomputedTransformationDataset(Dataset):
    """
    A standard PyTorch Dataset class tailored to serve the pre-computed data.

    This class acts as an interface between the pre-computed list of samples and
    the PyTorch DataLoader. It handles fetching individual samples by index and
    converting them into the appropriate PyTorch tensor formats required by the models.
    """
    def __init__(self, us_precomputed_data, mri_vols):
      """
        Initializes the dataset.

        Args:
            us_precomputed_data (list): The list generated by `create_precomputed_dataset`.
            mri_vols (list): The list of original 3D MRI volumes (NumPy arrays), kept in memory.
        """
      self.us_precomputed_data = us_precomputed_data
      self.mri_vols = mri_vols # Keep original MRIs in memory

    def __len__(self):
      """Returns the total number of samples in the dataset."""
      return len(self.us_precomputed_data)

    def __getitem__(self, idx):
      """
        Fetches a single sample from the dataset by its index.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing the data and labels for one sample, ready for model input:
                   (us_slice_tensor, mri_volume_tensor, label_s1, label_s2, label_s3, label_s4)
      """
      # Retrieve the pre-computed US slice, the MRI index, and labels for the requested index
      us_slice_np, mri_vol_idx, labels = self.us_precomputed_data[idx]
      # Retrieve the corresponding original 3D MRI volume using the stored index
      mri_volume_np = self.mri_vols[mri_vol_idx]

      # Convert the NumPy arrays to PyTorch tensors and ensure correct shape/type.
      # Add a channel dimension (C=1) for grayscale images.
      us_slice = torch.from_numpy(us_slice_np).float().unsqueeze(0)  # (C=1, H, W)
      # Add channel dimension (C=1) for the volume. Note PyTorch Conv3D expects (B, C, D, H, W).
      # We prepare (D, 1, H, W) here and will permute later if needed by the model.
      mri_volume = torch.from_numpy(mri_volume_np).float().unsqueeze(1) # (D, C=1, H, W)

      # Retrieve and format the labels for each stage.
      # Stage 1 label is a float tensor for regression.
      label_s1 = torch.from_numpy(labels["s1"])
      # Stages 2, 3, 4 labels are long tensors (integers) for classification.
      label_s2 = torch.tensor(labels["s2"], dtype=torch.long)
      label_s3 = torch.tensor(labels["s3"], dtype=torch.long)
      label_s4 = torch.tensor(labels["s4"], dtype=torch.long)

      # Return all components needed for training/validation
      return us_slice, mri_volume, label_s1, label_s2, label_s3, label_s4


# --- Model Building Blocks ---


def create_slice_encoder(feature_dim=FEATURE_DIM):
  """
    Factory function to create a 2D feature encoder based on ResNet-18.

    Loads a pre-trained ResNet-18 model, modifies its first convolutional layer
    to accept single-channel (grayscale) inputs, and replaces the final
    fully connected layer with a projection layer to output a feature vector
    of the specified dimension. Includes flattening and a ReLU activation.

    Args:
        feature_dim (int): The desired output dimensionality of the feature vector.

    Returns:
        torch.nn.Sequential: The constructed feature encoder model.
    """
  # Load a ResNet-18 model pre-trained on ImageNet
  resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
  # Modify the first convolutional layer (conv1) to accept 1 input channel instead of 3 (RGB).
  # Keep other parameters (kernel size, stride, padding) the same.
  resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
  # Get the number of input features to the original final fully connected layer (fc)
  num_ftrs = resnet.fc.in_features
  # Construct the final encoder by taking all layers *except* the last one (the classifier fc),
  # adding a Flatten layer, a Linear layer to project to the desired feature_dim,
  # and a ReLU activation.
  return nn.Sequential(
      *list(resnet.children())[:-1],
      nn.Flatten(),
      nn.Linear(num_ftrs, feature_dim),
      nn.ReLU()
  )

class AttentionSimilarityHead(nn.Module):
  """
    A custom neural network module implementing a cross-attention mechanism
    to compare spatial feature maps from two modalities (e.g., US and MRI).

    Takes two feature maps as input, computes attention scores to determine
    which parts of the MRI feature map are most relevant to the US feature map,
    weights the MRI features accordingly, and outputs a single similarity score.
    """
  def __init__(self, in_channels=512):
      """
        Initializes the layers needed for the attention mechanism.

        Args:
            in_channels (int): The number of channels in the input feature maps
                               (typically the output channels of the feature extractor).
      """
      super().__init__()
      # Layers to transform inputs into Query, Key, and Value
      # Query is derived from the US features.
      self.query = nn.Conv2d(in_channels, in_channels // 2, 1)
      # Key and Value are derived from the MRI features.
      self.key = nn.Conv2d(in_channels, in_channels // 2, 1)
      self.value = nn.Conv2d(in_channels, in_channels, 1)

      # Layers to process the attended features into a final score.
      # Adaptive average pooling reduces spatial dimensions to 1x1.
      self.pool = nn.AdaptiveAvgPool2d((1, 1))
      # Flatten the pooled output.
      self.flatten = nn.Flatten()
      # Final fully connected layer to output a single similarity score.
      self.fc = nn.Linear(in_channels, 1)

  def forward(self, us_features, mri_features):
      """
        Performs the forward pass of the attention mechanism.

        Args:
            us_features (torch.Tensor): Feature map from the US encoder (B, C, H_feat, W_feat).
            mri_features (torch.Tensor): Feature map from the MRI encoder (B, C, H_feat, W_feat).

        Returns:
            torch.Tensor: A tensor containing a single similarity score per batch item (B, 1).
        """
      # Get batch size, channels, height, width of the feature maps
      B, C, H, W = us_features.shape

      # 1. Generate Query, Key, Value representations.
      # Project features using the 1x1 conv layers.
      # Reshape spatial dimensions (H*W) into a single sequence dimension for matrix multiplication.
      q = self.query(us_features).view(B, -1, H * W)  # (B, C/2, H*W)
      k = self.key(mri_features).view(B, -1, H * W)    # (B, C/2, H*W)
      v = self.value(mri_features).view(B, -1, H * W)  # (B, C,   H*W)

      # 2. Calculate scaled dot-product attention scores.
      # Transpose Key for batch matrix multiplication (BMM): (B, SeqLen, C/2)
      # Multiply Query (transposed) with Key: (B, SeqLen, C/2) x (B, C/2, SeqLen) -> (B, SeqLen, SeqLen)
      # Scale by sqrt(dimensionality) to stabilize gradients.
      attention_scores = torch.bmm(q.transpose(1, 2), k) / (C**0.5)
      # Apply softmax along the Key dimension to get attention weights (probabilities summing to 1).
      attention_weights = torch.softmax(attention_scores, dim=-1) # (B, H*W, H*W)

      # 3. Apply attention weights to the Value vectors.
      # Multiply weights with Value (transposed): (B, SeqLen, SeqLen) x (B, SeqLen, C) -> (B, SeqLen, C)
      # This creates a weighted representation of the MRI features based on relevance to the US features.
      attended_mri = torch.bmm(attention_weights, v.transpose(1, 2)) # (B, H*W, C)

      # 4. Process the attended features to get the final score.
      # Transpose back and reshape to spatial feature map format: (B, C, H, W).
      attended_mri = attended_mri.transpose(1, 2).view(B, C, H, W)
      # Apply adaptive average pooling to get a single feature vector per sample.
      pooled = self.pool(attended_mri)
      # Flatten the result.
      flat = self.flatten(pooled)
      # Pass through the final linear layer to get the similarity score.
      return self.fc(flat)


class VolumeEncoder3D(nn.Module):
    """
    A lightweight 3D Convolutional Neural Network (CNN) designed to extract
    a fixed-size feature vector representing the global context of an entire
    3D MRI volume (e.g., 50x50x50).

    Uses a series of 3D convolutions, batch normalization, ReLU activations,
    and max pooling layers to progressively reduce spatial dimensions while
    increasing feature channels, followed by global average pooling and a
    final linear layer.
    """
    def __init__(self, feature_dim=256):
        """
        Initializes the layers of the 3D CNN.

        Args:
            feature_dim (int): The desired output dimensionality of the feature vector.
        """
        super().__init__()
        # Define the sequential convolutional blocks
        self.encoder = nn.Sequential(
            # Block 1: Input (B, 1, 50, 50, 50) -> Output (B, 16, 25, 25, 25)
            nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2), # (B, 16, 25, 25, 25)

            # Block 2: Input (B, 16, 25, 25, 25) -> Output (B, 32, 12, 12, 12)
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2), # (B, 32, 12, 12, 12)

            # Block 3: Input (B, 32, 12, 12, 12) -> Output (B, 64, 6, 6, 6)
            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2, stride=2), # (B, 64, 6, 6, 6)
        )
        # Global pooling layer reduces all spatial dimensions (D, H, W) to 1x1x1.
        self.pool = nn.AdaptiveAvgPool3d((1, 1, 1))
        # Flatten the output of the pooling layer.
        self.flatten = nn.Flatten()
        # Flatten the output of the pooling layer.
        self.fc = nn.Linear(64, feature_dim)

    def forward(self, mri_volume):
        """
        Performs the forward pass of the 3D encoder.

        Args:
            mri_volume (torch.Tensor): The input MRI volume tensor, expected to have shape
                                       (B, D, C, H, W) as provided by the DataLoader.

        Returns:
            torch.Tensor: The extracted feature vector of shape (B, feature_dim).
        """
        # PyTorch Conv3D expects input shape (B, C, D, H, W).
        # The DataLoader provides (B, D, C, H, W).
        # We permute the dimensions to match the Conv3D expectation.
        # (0, 2, 1, 3, 4) maps (B, D, C, H, W) -> (B, C, D, H, W)
        x = mri_volume.permute(0, 2, 1, 3, 4)
        # Pass the permuted volume through the convolutional blocks
        x = self.encoder(x)
        # Apply global average pooling
        x = self.pool(x)
        # Flatten the result
        x = self.flatten(x)
        # Pass through the final fully connected layer
        x = self.fc(x)
        # Return the final feature vector
        return x


class NCCLoss(nn.Module):
    """
    A custom, self-contained implementation of the Normalized Cross-Correlation (NCC) loss function.

    Calculates the NCC between two input tensors (e.g., images or feature maps)
    and returns `1 - NCC` as the loss value. Minimising this loss maximizes the
    similarity (NCC value) between the inputs. This implementation uses only
    standard PyTorch operations and does not depend on specific versions of external
    libraries like Kornia.

    NCC is robust to linear changes in brightness and contrast between the inputs.
    """
    def __init__(self, epsilon=1e-6):
        """
        Initializes the loss function.

        Args:
            epsilon (float): A small value added to the denominator for numerical stability
                             to prevent division by zero.
        """
        super().__init__()
        self.epsilon = epsilon

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Calculates the NCC loss between the input and target tensors.

        Args:
            input (torch.Tensor): The first input tensor (e.g., predicted image),
                                   expected shape (B, C, H, W).
            target (torch.Tensor): The second input tensor (e.g., ground truth image),
                                    expected shape (B, C, H, W).

        Returns:
            torch.Tensor: A scalar tensor representing the average NCC loss (1 - mean(NCC))
                          across the batch.
        """
        # Get the shape of the input tensors (Batch, Channels, Height, Width)
        B, C, H, W = input.shape

        # Flatten the spatial dimensions (C, H, W) into a single vector per batch item.
        # This treats each image/feature map as a long vector for correlation calculation.
        input_flat = input.view(B, -1)
        target_flat = target.view(B, -1)

        # --- NCC Calculation Steps ---
        # 1. Compute the mean intensity/value for each flattened vector in the batch.
        #    keepdim=True maintains the dimension for broadcasting during subtraction.
        input_mean = torch.mean(input_flat, dim=1, keepdim=True)
        target_mean = torch.mean(target_flat, dim=1, keepdim=True)

        # 2. Center the data by subtracting the mean (zero-mean normalization).
        #    This makes NCC robust to global brightness shifts.
        input_centered = input_flat - input_mean
        target_centered = target_flat - target_mean

        # 3. Calculate the cross-correlation numerator: Sum of element-wise products of centered vectors.
        #    Summing over dim=1 collapses the vector dimension.
        corr_numerator = torch.sum(input_centered * target_centered, dim=1)

        # 4. Calculate the auto-correlation terms for the denominator (related to standard deviation).
        std_input = torch.sqrt(torch.sum(input_centered**2, dim=1))
        std_target = torch.sqrt(torch.sum(target_centered**2, dim=1))
        corr_denominator = std_input * std_target

        # 5. Compute the NCC value for each item in the batch.
        #    Add epsilon to the denominator for numerical stability (prevents division by zero).
        ncc = corr_numerator / (corr_denominator + self.epsilon)
        # Average the NCC values across the batch to get a single scalar value.
        mean_ncc = torch.mean(ncc)

        # 6. Return the loss value: 1 - mean(NCC).
        #    Since NCC ranges from -1 to 1 (higher is better similarity),
        #    1 - NCC ranges from 0 to 2 (lower is better for loss minimization).
        return 1 - mean_ncc

# --- 4. Model Definitions ---

In [ ]:
# --- 4. Model Definitions ---

class CoupledPredictor(nn.Module):
    """
    Stage 1 Model: Predicts 3DOF using a dedicated 3D Volume Encoder.

    Uses separate encoders: a 2D encoder for the US slice and a 3D CNN encoder
    for the entire MRI volume, followed by late-stage fusion and regression.
    """
    def __init__(self, feature_dim=256, num_params=3):
        """
        Initializes the CoupledPredictor model layers.

        Args:
            feature_dim (int): Dimensionality of the feature vectors from both encoders.
            num_params (int): Number of parameters to regress (e.g., 3 for Z, X, Y).
        """
        super().__init__()
        # Dedicated 2D encoder for the US slice
        self.slice_encoder = create_slice_encoder(feature_dim)
        # Dedicated 3D encoder for the MRI volume
        self.volume_encoder = VolumeEncoder3D(feature_dim)

        # Regression head: Takes concatenated features and predicts mean (mu) and log-variance (log_var)
        self.regression_head = nn.Sequential(
            nn.Linear(feature_dim * 2, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, num_params * 2) # mu and log_var for each param
        )

    def forward(self, us_slice, mri_volume):
        """
        Performs the forward pass using separate 2D and 3D encoders.

        Args:
            us_slice (torch.Tensor): Input US slice tensor (B, 1, H, W).
            mri_volume (torch.Tensor): Input MRI volume tensor (B, D, 1, H, W).

        Returns:
            tuple: (mu, var)
                   - mu (torch.Tensor): Predicted mean values for Z, X, Y (B, num_params).
                   - var (torch.Tensor): Predicted variance values for Z, X, Y (B, num_params).
        """
        # --- Branch 1: Process US Slice ---
        # Extract features from the 2D US slice using the 2D encoder
        us_features = self.slice_encoder(us_slice)

        # --- Branch 2: Process MRI Volume ---
        # Extract features from the entire 3D MRI volume using the 3D encoder
        mri_context_features = self.volume_encoder(mri_volume)

        # --- Fusion and Regression ---
        # Concatenate the features from the two branches
        combined_features = torch.cat((us_features, mri_context_features), dim=1)
        # Pass through the regression head
        output = self.regression_head(combined_features)

        # --- Output Processing ---
        # Split into mean (mu) and log-variance (log_var)
        mu, log_var = output[:, :3], output[:, 3:]
        # Calculate variance
        var = torch.exp(log_var)
        # Return predicted mean and variance
        return mu, var


class ZRefiner(nn.Module):
    """
    Stage 2 Model: Predicts the Z-slice index via classification.

    This model only uses the US slice as input to predict which of the possible
    Z-slices it corresponds to. It does not directly compare against the MRI volume
    during inference, relying instead on features learned during training.
    """

    def __init__(self, feature_dim=FEATURE_DIM):
        """
        Initializes the ZRefiner model layers.

        Args:
            feature_dim (int): Dimensionality of the feature vector from the encoder.
        """
        super().__init__()
        # 2D feature encoder for the US slice
        self.slice_encoder = create_slice_encoder(feature_dim)
        # Classification head: Takes US features and outputs scores for each Z-slice.
        self.similarity_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.ReLU(),
            nn.Linear(256, VOLUME_DIM) # Direct classification over all slices
        )

    def forward(self, us_slice, mri_volume):
        """
        Performs the forward pass for Z-slice classification.

        Args:
            us_slice (torch.Tensor): Input US slice tensor (B, 1, H, W).
            mri_volume (torch.Tensor): Input MRI volume tensor (B, D, 1, H, W).

        Returns:
            torch.Tensor: Logits (scores) for each possible Z-slice (B, VOLUME_DIM).
        """
        # Extract features from the US slice
        us_features = self.slice_encoder(us_slice)
        # Pass features through the classification head to get scores for each Z-slice
        return self.similarity_head(us_features)


class RotationRefiner(nn.Module):
    """
    Stage 3/4 Model (Pseudo-Siamese): Compares US slice to MRI slice.

    Uses SEPARATE feature extractors for US and MRI inputs, allowing each branch
    to specialize in features relevant to its modality.
    """

    def __init__(self, rotation_options):
        """
        Initializes the RotationRefinerV2 model layers.

        Args:
            rotation_options (iterable): List/array of discrete rotation angles.
        """
        super().__init__()

        # --- Separate Feature Extractors (Pseudo-Siamese) ---
        # Helper function to create one instance of the ResNet-based feature extractor
        def _create_feature_extractor():
            resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            return nn.Sequential(*list(resnet.children())[:-2])

        # Create two distinct instances of the extractor. They will have different weights.
        self.us_feature_extractor = _create_feature_extractor()
        self.mri_feature_extractor = _create_feature_extractor()

        # Comparison head and rotation options
        self.similarity_head = AttentionSimilarityHead(in_channels=512)
        self.register_buffer('all_rotation_options', torch.tensor(rotation_options, dtype=torch.float32))

    def forward(self, us_slice, mri_slice, angles_to_test=None):
        """
        Performs the forward pass using separate feature extractors.

        Args:
            us_slice (torch.Tensor): Input US slice tensor (B, 1, H, W).
            mri_slice (torch.Tensor): Input candidate MRI slice tensor (B, 1, H, W).
            angles_to_test (torch.Tensor, optional): Specific angles to test (B, num_guided). Defaults to None.

        Returns:
            torch.Tensor: Scores for each tested rotation angle (B, num_angles_tested).
        """
        B = us_slice.shape[0]

        # --- Feature Extraction (Separate Weights) ---
        # Pass each input through its dedicated feature extractor instance.
        us_features = self.us_feature_extractor(us_slice)
        mri_features = self.mri_feature_extractor(mri_slice)

        # --- Determine Angles to Test ---
        all_scores = [] # List to store scores for each tested angle
        # If specific angles are provided (guided mode), use them.
        # Otherwise (unguided mode or training), use the full list of angles.
        angle_source = self.all_rotation_options if angles_to_test is None else angles_to_test
        num_angles_to_test = angle_source.shape[1] if angles_to_test is not None else len(angle_source)

        # --- Rotation and Comparison Loop ---
        # Iterate through the angles that need to be tested
        # Guided: Extract the i-th column of angles, one specific angle per batch item
        # Unguided: Use the i-th angle from the static list for all items in the batch
        for i in range(num_angles_to_test):
            if angles_to_test is not None:
                angle_batch = angle_source[:, i]
            else:
                angle_batch = angle_source[i].expand(B)

            # --- Rotate MRI Features ---
            # Apply the rotation to the MRI *feature map* using Kornia (GPU-accelerated)
            rotated_mri_features = K.rotate(mri_features, angle_batch)

            # --- Compare Features ---
            # Pass the US features (Query) and rotated MRI features (Key, Value) to the attention head
            score = self.similarity_head(us_features, rotated_mri_features)
            all_scores.append(score)

        # Concatenate the scores for all tested angles along dimension 1
        return torch.cat(all_scores, dim=1)

# --- 5. Training & Validation Logic ---

In [ ]:
# --- 5. Training & Validation Logic ---

def train_one_epoch(model, dataloader, criterion, optimizer, stage, models_prev=None):
    """
    Runs a single training epoch using the standard loss function (e.g., NLLLoss, CrossEntropy).
    Used for Stages 1 and 2, and potentially S3/S4 if hybrid loss is not used.

    Args:
        model (nn.Module): The model for the current stage being trained.
        dataloader (DataLoader): DataLoader providing training data batches.
        criterion (nn.Module): The loss function (e.g., GaussianNLLLoss, CrossEntropyLoss).
        optimizer (optim.Optimizer): The optimizer (e.g., Adam).
        stage (int): The current training stage number (1, 2, 3, or 4).
        models_prev (dict, optional): Dictionary of previously trained models, used if
                                     the current stage needs input from earlier stages
                                     Defaults to None.

    Returns:
        float: The average training loss over all batches in the epoch.
    """
    # Set the current model to training mode (enables dropout, batchnorm updates)
    model.train()
    # If previous stage models are provided, set them to evaluation mode
    if models_prev:
        for m in models_prev.values(): m.eval()

    total_loss = 0.0
    # Iterate through batches provided by the DataLoader
    for data in tqdm(dataloader, desc=f"Training S{stage}", leave=False):
        # Move all data tensors to the configured device (CPU or GPU)
        us_slice, mri_volume, label_s1, label_s2, label_s3, label_s4 = [d.to(DEVICE) for d in data]
        # Reset gradients accumulated from the previous iteration
        optimizer.zero_grad()

        # --- Stage-Specific Forward Pass and Loss Calculation ---
        # Select the appropriate model input and ground truth label based on the stage.
        # Note: S3 and S4 use ground truth inputs from previous stages during training
        #       for stability and direct supervision.

        if stage == 1: # Coarse Regression
            # Get mean (mu) and variance (var) predictions from the model
            mu, var = model(us_slice, mri_volume)
            # Calculate Gaussian Negative Log Likelihood loss using mu, var, and true S1 labels
            loss = criterion(mu, label_s1, var)

        elif stage == 2: # Z-Slice Classification
            # Get classification scores (logits) from the model
            outputs = model(us_slice, mri_volume)
            # Calculate Cross Entropy loss using scores and true Z-slice index (S2 label)
            loss = criterion(outputs, label_s2)

        elif stage == 3: # X-Rotation Classification
            # Extract the MRI slice corresponding to the Z-index (label_s2)
            mri_slice_true_z = mri_volume[torch.arange(len(label_s2)), label_s2, ...]
            # Get classification scores using the US slice and the ground truth Z-aligned MRI slice
            outputs = model(us_slice, mri_slice_true_z)
            # Calculate Cross Entropy loss using scores and true X-rotation index (S3 label)
            loss = criterion(outputs, label_s3)

        elif stage == 4: # Y-Rotation Classification
            # Extract the MRI slice corresponding to the Z-index (label_s2)
            mri_slice_true_z = mri_volume[torch.arange(len(label_s2)), label_s2, ...]
            # Convert X-rotation indices (label_s3) to angles
            true_x_angles = torch.tensor([idx_to_x_rot[i.item()] for i in label_s3], device=DEVICE)
            # Rotate the Z-aligned MRI slice by the X-angle
            mri_slice_true_zx = K.rotate(mri_slice_true_z, true_x_angles)
            # Get classification scores using the US slice and the Z/X-aligned MRI slice
            outputs = model(us_slice, mri_slice_true_zx)
            # Calculate Cross Entropy loss using scores and true Y-rotation index (S4 label)
            loss = criterion(outputs, label_s4)

        # --- Backpropagation and Optimisation ---
        # Calculate gradients of the loss with respect to model parameters
        loss.backward()
        # Update model parameters based on the calculated gradients
        optimizer.step()
        # Accumulate the loss for this batch (converting tensor to float)
        total_loss += loss.item()

    # Calculate the average loss over the entire epoch
    return total_loss / len(dataloader)


def train_one_epoch_hybrid_loss(
    model, dataloader, criterion_cls, optimizer, stage,
    lambda_ncc=0.5, models_prev=None
):
    """
    Runs a single training epoch for Stage 3 or Stage 4 using their HYBRID loss function.
    The total loss is a weighted sum of CrossEntropy (classification) and NCC (similarity).
    Loss = CrossEntropy + lambda_ncc * (1 - NCC_Similarity).

    Args:
        model (nn.Module): The RotationRefiner model (S3 or S4).
        dataloader (DataLoader): DataLoader providing training data.
        criterion_cls (nn.Module): The Cross Entropy loss function.
        optimizer (optim.Optimizer): The optimizer.
        stage (int): The current stage (should be 3 or 4).
        lambda_ncc (float): Weighting factor for the NCC loss component. Defaults to 0.5.
        models_prev (dict, optional): Dictionary of previously trained models. Defaults to None.

    Returns:
        float: The average total hybrid training loss over the epoch.
    """

    # Set the current model to training mode
    model.train()
    # Set previous models (if any) to evaluation mode
    if models_prev:
        for m in models_prev.values(): m.eval()

    # Get the device (CPU/GPU) the model's parameters are currently on
    # This ensures tensors created later are on the same device
    device = next(model.parameters()).device

    # Instantiate the custom NCC Loss function
    criterion_ncc = NCCLoss()

    total_loss = 0.0 # Accumulator for the total hybrid loss

    # Iterate through training batches
    for data in tqdm(dataloader, desc=f"Training S{stage} (Hybrid)", leave=False):

        # Move data to the correct device
        us_slice, mri_volume, label_s1, label_s2, label_s3, label_s4 = [d.to(device) for d in data]

        # Reset gradients
        optimizer.zero_grad()
        loss = 0 # Initialize loss for this batch

        # --- Stage-Specific Input Preparation and Loss Calculation ---
        # Uses ground truth inputs from previous stages for stability during training.

        if stage == 3: # X-Rotation Refinement
            # --- Input Preparation ---
            # Get the ground truth Z-slice index
            true_z_idx = label_s2
            # Extract the corresponding MRI slice from the volume
            mri_slice_input = mri_volume[torch.arange(len(label_s2)), true_z_idx, ...]

            # --- Forward Pass ---
            # Get the classification scores (logits) for the 5 possible X-angles
            outputs = model(us_slice, mri_slice_input) # (B, 5) scores

            # --- Hybrid Loss Part 1: Cross Entropy ---
            # Calculate the standard classification loss
            loss_cls = criterion_cls(outputs, label_s3)
            loss += loss_cls

            # --- Hybrid Loss Part 2: NCC Similarity ---
            # Get the predicted angle index (use detach() to prevent gradients flowing back through argmax)
            pred_indices = torch.argmax(outputs.detach(), dim=1)
            # Convert the predicted indices back to angle values
            pred_angles = torch.tensor([idx_to_x_rot[i.item()] for i in pred_indices], device=device)
            # Rotate the input MRI slice according to the predicted angle
            rotated_mri_pred = K.rotate(mri_slice_input, pred_angles)
            # Calculate the NCC loss (1 - Similarity) between the US slice and the predicted-rotated MRI slice
            loss_ncc = criterion_ncc(us_slice, rotated_mri_pred)
            # Add the weighted NCC loss to the total batch loss
            loss += lambda_ncc * loss_ncc

        elif stage == 4: # Y-Rotation Refinement
            # --- Input Preparation ---
            # Extract the MRI slice at the ground truth Z-index
            mri_slice_true_z = mri_volume[torch.arange(len(label_s2)), label_s2, ...]
            # Convert the ground truth X-rotation indices to angles
            true_x_angles = torch.tensor([idx_to_x_rot[i.item()] for i in label_s3], device=device)
            # Rotate the MRI slice by the ground truth X-angle to prepare the input for S4
            mri_slice_input = K.rotate(mri_slice_true_z, true_x_angles)

            # --- Forward Pass ---
            # Get the classification scores (logits) for the 5 possible Y-angles
            outputs = model(us_slice, mri_slice_input)

            # --- Hybrid Loss Part 1: Cross Entropy ---
            # Calculate the standard classification loss
            loss_cls = criterion_cls(outputs, label_s4)
            loss += loss_cls

            # --- Hybrid Loss Part 2: NCC Similarity ---
            # Get the predicted angle index
            pred_indices = torch.argmax(outputs.detach(), dim=1)
            # Convert the predicted indices back to angle values
            pred_angles = torch.tensor([idx_to_y_rot[i.item()] for i in pred_indices], device=device)
            # Rotate the input MRI slice (already Z/X aligned) by the predicted Y-angle
            rotated_mri_pred = K.rotate(mri_slice_input, pred_angles)
            # Calculate the NCC loss between the US slice and the fully predicted-rotated MRI slice
            loss_ncc = criterion_ncc(us_slice, rotated_mri_pred)
            # Add the weighted NCC loss to the total batch loss
            loss += lambda_ncc * loss_ncc

        # --- Backpropagation and Optimization ---
        # Calculate gradients for the total hybrid loss
        loss.backward()
        # Update model parameters
        optimizer.step()
        # Accumulate the total loss for the epoch
        total_loss += loss.item()

    # Return the average total hybrid loss over the epoch
    return total_loss / len(dataloader)



def validate_stage(model, dataloader, criterion, stage, models_prev=None):

    """
    Runs a single validation epoch for a given stage. Calculates loss and metrics (MAE, Acc).
    Includes logic to calculate the hybrid loss for S3/S4 for consistent plotting.

    Args:
        model (nn.Module): The model for the current stage being validated.
        dataloader (DataLoader): DataLoader providing validation data batches.
        criterion (nn.Module): The primary loss function (NLLLoss or CrossEntropy).
        stage (int): The current validation stage number (1, 2, 3, or 4).
        models_prev (dict, optional): Previously trained models (not used here as validation uses GT inputs).
                                      Defaults to None.

    Returns:
        tuple: (avg_loss, mae, acc)
               - avg_loss (float): Average loss over the validation set.
               - mae (torch.Tensor or float): Mean Absolute Error (multi-dim for S1, scalar otherwise).
               - acc (torch.Tensor or float): Accuracy (multi-dim for S1, scalar otherwise).
    """
    # Set the model to evaluation mode (disables dropout, fixes batchnorm)
    model.eval()
    # Set previous models to eval mode
    if models_prev:
        for m in models_prev.values(): m.eval()

    total_loss = 0.0 # Accumulator for loss
    all_preds_raw, all_labels_raw = [], [] # List to store raw model outputs (logits or mu)
    # and list to store corresponding ground truth labels

    # If validating S3 or S4, instantiate the NCCLoss for hybrid calculation
    if stage in [3, 4]:
        criterion_ncc = NCCLoss()
        lambda_ncc = 0.5

    # Disable gradient calculations during validation for efficiency
    with torch.no_grad():
        # Iterate through validation batches
        for data in tqdm(dataloader, desc=f"Validating S{stage}", leave=False):
            # Move data to device
            us_slice, mri_volume, label_s1, label_s2, label_s3, label_s4 = [d.to(DEVICE) for d in data]

            # --- Stage-Specific Forward Pass and Loss Calculation (using Ground Truth inputs) ---
            if stage == 1: # Coupled Regression
                # Get mean (mu) and variance (var) predictions
                mu, var = model(us_slice, mri_volume)
                # Ground truth labels for S1
                labels = label_s1
                # Calculate NLL loss
                loss = criterion(mu, labels, var)
                # Store the predicted mean values (raw output)
                all_preds_raw.append(mu.cpu())
                # Store the ground truth labels
                all_labels_raw.append(labels.cpu())

            else: # Classification Stages (S2, S3, S4)
                # Get the correct label for the current classification stage
                labels = [label_s2, label_s3, label_s4][stage - 2]

                if stage == 2: # Z-Slice Classification
                    # Get classification scores
                    outputs = model(us_slice, mri_volume)
                    # Calculate standard Cross Entropy loss
                    loss = criterion(outputs, labels)

                elif stage == 3: # X-Rotation Classification (Calculate Hybrid Loss)
                    # Extract MRI slice at Z-index
                    mri_slice_true_z = mri_volume[torch.arange(len(label_s2)), label_s2, ...]
                    # Get classification scores
                    outputs = model(us_slice, mri_slice_true_z)

                    # --- Hybrid Loss Calculation (Mirroring Training) ---
                    # Calculate Cross Entropy part
                    loss_cls = criterion(outputs, labels)

                    # Get predicted angle from scores
                    pred_indices = torch.argmax(outputs.detach(), dim=1)
                    pred_angles = torch.tensor([idx_to_x_rot[i.item()] for i in pred_indices], device=DEVICE)

                    # Rotate the input slice by the predicted angle
                    rotated_mri_pred = K.rotate(mri_slice_true_z, pred_angles)
                    # Calculate NCC part
                    loss_ncc = criterion_ncc(us_slice, rotated_mri_pred)
                    # Combine for the total hybrid loss
                    loss = loss_cls + lambda_ncc * loss_ncc

                elif stage == 4: # Y-Rotation Classification (Calculate Hybrid Loss)
                    # Prepare the input slice using GROUND TRUTH Z and X
                    mri_slice_true_z = mri_volume[torch.arange(len(label_s2)), label_s2, ...]
                    true_x_angles = torch.tensor([idx_to_x_rot[i.item()] for i in label_s3], device=DEVICE)
                    mri_slice_true_zx = K.rotate(mri_slice_true_z, true_x_angles)
                    # Get classification scores
                    outputs = model(us_slice, mri_slice_true_zx)

                    # --- Hybrid Loss Calculation (Mirroring Training) ---
                    # Calculate Cross Entropy part
                    loss_cls = criterion(outputs, labels)
                    # Get predicted angle from scores
                    pred_indices = torch.argmax(outputs.detach(), dim=1)
                    pred_angles = torch.tensor([idx_to_y_rot[i.item()] for i in pred_indices], device=DEVICE)
                    # Rotate the input slice by the predicted angle
                    rotated_mri_pred = K.rotate(mri_slice_true_zx, pred_angles)
                    # Calculate NCC part
                    loss_ncc = criterion_ncc(us_slice, rotated_mri_pred)
                    # Combine for the total hybrid loss
                    loss = loss_cls + lambda_ncc * loss_ncc

                # Store the raw classification scores (logits)
                all_preds_raw.append(outputs.cpu())
                # Store the ground truth labels
                all_labels_raw.append(labels.cpu())

            # Accumulate the loss for this batch
            total_loss += loss.item()

    # --- Post-Epoch Calculations ---
    # Calculate average loss over the validation set
    avg_loss = total_loss / len(dataloader)
    # Concatenate all stored predictions and labels into single tensors
    preds_cat = torch.cat(all_preds_raw)
    labels_cat = torch.cat(all_labels_raw)

    # --- Calculate Metrics (MAE and Accuracy) ---
    if stage == 1:
        # Calculate absolute error for each parameter (Z, X, Y)
        error = torch.abs(preds_cat - labels_cat)
        # Calculate Mean Absolute Error for each parameter
        mae = error.mean(dim=0)
        # Calculate accuracy based on tolerance thresholds
        acc_z = (error[:, 0] < 1.0).float().mean() * 100
        acc_x = (error[:, 1] < 2.5).float().mean() * 100
        acc_y = (error[:, 2] < 2.5).float().mean() * 100
        # Combine accuracies into a tensor
        acc = torch.tensor([acc_z, acc_x, acc_y])

    else:
        # Get the predicted class index by taking the argmax of the scores
        pred_indices = torch.argmax(preds_cat, dim=1)
        # Calculate accuracy as the percentage of correct predictions
        acc = (pred_indices == labels_cat).float().mean() * 100

        # Calculate MAE in original units (slices or degrees)
        if stage == 2:
            mae = torch.abs(pred_indices - labels_cat).float().mean()
        elif stage == 3:
            pred_angles = torch.tensor([idx_to_x_rot[i.item()] for i in pred_indices])
            true_angles = torch.tensor([idx_to_x_rot[i.item()] for i in labels_cat])
            mae = torch.abs(pred_angles - true_angles).float().mean()
        elif stage == 4:
            pred_angles = torch.tensor([idx_to_y_rot[i.item()] for i in pred_indices])
            true_angles = torch.tensor([idx_to_y_rot[i.item()] for i in labels_cat])
            mae = torch.abs(pred_angles - true_angles).float().mean()

    # Return the calculated average loss, MAE, and accuracy
    return avg_loss, mae, acc

def run_training_stage(stage_num, stage_name, model, criterion_train, criterion_val,
                       optimizer, num_epochs, train_loader, val_loader,
                       models_prev=None, use_hybrid_loss=False, **kwargs):
    """
    Manages the training and validation loop for a single stage over multiple epochs.

    This function acts as a wrapper, calling the appropriate epoch-level training
    (`train_one_epoch` or `train_one_epoch_hybrid_loss`) and validation (`validate_stage`)
    functions for the specified number of epochs. It logs the history of loss and metrics.

    Args:
        stage_num (int): The current stage number (1-4).
        stage_name (str): A descriptive name for the stage (e.g., "Coarse Regression").
        model (nn.Module): The model for this stage.
        criterion_train (nn.Module): The loss function to use during training.
        criterion_val (nn.Module): The loss function to use during validation.
        optimizer (optim.Optimizer): The optimizer for the model.
        num_epochs (int): The total number of epochs to run for this stage.
        train_loader (DataLoader): DataLoader for training data.
        val_loader (DataLoader): DataLoader for validation data.
        models_prev (dict, optional): Dictionary of previously trained models. Defaults to None.
        use_hybrid_loss (bool): Flag indicating whether to use the hybrid loss function
                                 for training and validation (applies to S3/S4). Defaults to False.
        **kwargs: Additional keyword arguments to pass to the training function
                  (e.g., `lambda_ncc` for hybrid loss).

    Returns:
        tuple: (model, history)
               - model (nn.Module): The trained model for this stage.
               - history (dict): A dictionary containing lists of training loss, validation loss,
                                 validation MAE, and validation accuracy for each epoch.
    """

    print(f"\n--- Training Stage {stage_num}: {stage_name} ---")
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_acc': []}

    for epoch in range(num_epochs):
        # --- Training Step ---
        # Decide which training function to call based on the 'use_hybrid_loss' flag
        if use_hybrid_loss:
            # Call the hybrid loss training function, passing extra arguments via **kwargs
            train_loss = train_one_epoch_hybrid_loss(
                model, train_loader, criterion_train, optimizer, stage=stage_num,
                models_prev=models_prev, **kwargs
            )
        else:
            # Call the standard training function (for S1/S2)
            train_loss = train_one_epoch(
                model, train_loader, criterion_train, optimizer, stage=stage_num, models_prev=models_prev
            )

        # --- Validation Step ---
        # Call the validation function (which now internally handles hybrid loss calculation if needed)
        val_loss, val_mae, val_acc = validate_stage(
            model, val_loader, criterion_val, stage=stage_num, models_prev=models_prev
        )

        # --- Logging ---
        # Append the results for the current epoch to the history dictionary
        # Convert tensors to NumPy arrays for easier handling/plotting later
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_mae'].append(val_mae.cpu().numpy())
        history['val_acc'].append(val_acc.cpu().numpy())

        # --- Print Epoch Summary ---
        # Display performance metrics for the current epoch
        if stage_num == 1:
            print(f"Epoch {epoch+1}/{num_epochs} -> Train Loss: {train_loss:.4f} | Val MAE: [Z:{val_mae[0]:.2f}, X:{val_mae[1]:.2f}, Y:{val_mae[2]:.2f}] | Val Acc: [Z:{val_acc[0]:.1f}%, X:{val_acc[1]:.1f}%, Y:{val_acc[2]:.1f}%]")
        else:
            print(f"Epoch {epoch+1}/{num_epochs} -> Train Loss: {train_loss:.4f} | Val MAE: {val_mae:.3f} | Val Acc: {val_acc:.2f}%")

    # Return the trained model and the recorded history
    return model, history

# --- 6. Evaluation & Inference Logic ---

In [ ]:
# --- 6. Evaluation & Inference Logic ---

def evaluate_full_pipeline(models, dataloader):
    """
    Evaluates the end-to-end performance of the UNGUIDED sequential pipeline (S2 -> S3 -> S4).

    This function simulates the scenario where the coarse prediction (S1) is NOT used.
    It feeds the output prediction from one stage directly as input to the next,
    taking the highest-scoring option (argmax) at each classification step without any restriction.
    It calculates the final Mean Absolute Error (MAE) and average inference time per slice.

    Args:
        models (dict): Dictionary containing the trained models for each stage ('s1', 's2', 's3', 's4').
        dataloader (DataLoader): DataLoader providing the test dataset batches.

    Returns:
        tuple: (mae, preds_np, labels_np)
               - mae (np.ndarray): Array containing MAE for Z, X, Y.
               - preds_np (np.ndarray): NumPy array of all final predictions (N, 3).
               - labels_np (np.ndarray): NumPy array of all corresponding ground truth labels (N, 3).
    """

    # Set all models to evaluation mode
    for m in models.values(): m.eval()

    # Lists to accumulate predictions and ground truth labels across all batches
    all_final_preds, all_true_labels = [], []
    # Variables for timing
    total_time, num_samples = 0, 0

    with torch.no_grad():
        # Iterate through batches from the test dataloader
        for data in tqdm(dataloader, desc="Unguided Pipeline Evaluation"):

            # Move data to the designated device
            us_slice, mri_volume, label_s1, _, _, _ = [d.to(DEVICE) for d in data]
            B = us_slice.shape[0]

            # Start timer for this batch's inference
            start_time = time.perf_counter()

            # --- Stage 2: Refine Z (Unguided) ---
            # Predict scores for all possible Z slices
            z_scores = models['s2'](us_slice, mri_volume)
            z_pred_idx = torch.argmax(z_scores, dim=1)

            # --- Stage 3: Refine X (Unguided) ---
            # Extract the MRI slice corresponding to the *predicted* Z index from Stage 2
            mri_slice_pred_z = mri_volume[torch.arange(B), z_pred_idx, ...]
            # Predict scores for all possible X rotation angles using the S2 output slice
            x_scores = models['s3'](us_slice, mri_slice_pred_z)
            # Select the X rotation index with the highest score (no masking/guidance)
            x_pred_idx = torch.argmax(x_scores, dim=1)

            # Stage 4: Refine Y
            # Convert the *predicted* X rotation index from Stage 3 to an angle value
            x_pred_angles = torch.tensor([idx_to_x_rot[i.item()] for i in x_pred_idx], device=DEVICE)
            # Rotate the S2-predicted MRI slice by the S3-predicted X angle
            mri_slice_pred_zx = K.rotate(mri_slice_pred_z, x_pred_angles)
            # Predict scores for all possible Y rotation angles using the S2/S3 output slice
            y_scores = models['s4'](us_slice, mri_slice_pred_zx)
            # Select the Y rotation index with the highest score (no masking/guidance)
            y_pred_idx = torch.argmax(y_scores, dim=1)

            # Stop timer for this batch
            total_time += (time.perf_counter() - start_time)
            num_samples += B

            # --- Collate Results for this Batch ---
            # Convert final predicted indices to physical units (slices and degrees)
            z_preds = z_pred_idx.cpu().numpy()
            x_preds = np.array([idx_to_x_rot[i.item()] for i in x_pred_idx.cpu()])
            y_preds = np.array([idx_to_y_rot[i.item()] for i in y_pred_idx.cpu()])

            # Stack the predictions for Z, X, Y into a single array for the batch
            final_preds_batch = np.stack([z_preds, x_preds, y_preds], axis=1)

            # Append batch predictions and ground truths (from label_s1) to the lists
            all_final_preds.append(final_preds_batch)
            all_true_labels.append(label_s1.cpu().numpy())

    # --- Aggregate Results Across All Batches ---
    # Concatenate predictions and labels from all batches into single large NumPy arrays
    preds_np = np.concatenate(all_final_preds)
    labels_np = np.concatenate(all_true_labels)

    # Calculate Mean Absolute Error (MAE) for each parameter (Z, X, Y)
    mae = np.mean(np.abs(preds_np - labels_np), axis=0)
    # Calculate average inference time per slice in milliseconds
    avg_time_ms = (total_time / num_samples) * 1000

    # Print summary results
    print("\n--- Unguided Pipeline End-to-End Performance ---")
    print(f"MAE Z: {mae[0]:.3f} slices | MAE X-Rot: {mae[1]:.3f}° | MAE Y-Rot: {mae[2]:.3f}°")
    print(f"Average Inference Time: {avg_time_ms:.2f} ms per slice")

    # Return calculated metrics and raw predictions/labels
    return mae, preds_np, labels_np


def get_guided_prediction(models, us_slice_batch, mri_volume_batch, confidence_multiplier=1.0):
    """
    Runs the full S1->S2->S3->S4 GUIDED pipeline for a single batch of data.

    This function implements the core logic for guided inference:
    1. Uses S1 (CoarsePredictor) to get an initial prediction and uncertainty.
    2. Uses S1 uncertainty to create a dynamic search window for S2 (Z-refinement).
    3. Uses S1 prediction to select the 2 most likely angles for S3 (X-refinement)
       and S4 (Y-refinement), enabling efficient "active guidance".

    Args:
        models (dict): Dictionary of trained models ('s1', 's2', 's3', 's4').
        us_slice_batch (torch.Tensor): Batch of input US slices (B, 1, H, W).
        mri_volume_batch (torch.Tensor): Batch of input MRI volumes (B, D, 1, H, W).
        confidence_multiplier (float): Multiplier for the S1 standard deviation (sigma)
                                       to determine the Z-search window radius. Defaults to 1.0.

    Returns:
        tuple: (z_pred_idx, x_pred_idx, y_pred_idx)
               Predicted indices for Z-slice, X-rotation, and Y-rotation for the batch.
    """
    B = us_slice_batch.shape[0]
    dev = us_slice_batch.device

    # Get all possible rotation angles as tensors
    x_rots_all = models['s3'].all_rotation_options.to(dev)
    y_rots_all = models['s4'].all_rotation_options.to(dev)

    # S1: Coupled prediction - capture both mu and var
    mu_s1, var_s1 = models['s1'](us_slice_batch, mri_volume_batch)

    # --- Stage 2: Z-Refinement with a dynamic window ---
    z_scores_full = models['s2'](us_slice_batch, mri_volume_batch)
    z_mask = torch.full_like(z_scores_full, -float('inf'))

    # 1. Calculate the standard deviation for the Z prediction for each item in the batch
    # sigma = sqrt(variance). We only need the first element (Z-dimension).
    sigma_z = torch.sqrt(var_s1[:, 0])

    # 2. Calculate a dynamic radius for each sample based on its uncertainty
    # We round it to the nearest integer to use as an index.
    z_window_radius_dynamic = torch.round(sigma_z * confidence_multiplier).long()

    # 3. Apply the unique, dynamic mask for each sample in the batch
    for i in range(B):
        z_center = torch.round(mu_s1[i, 0]).long()
        radius = z_window_radius_dynamic[i]

        # Ensure the window stays within the volume's bounds [0, 49]
        start_idx = torch.clamp(z_center - radius, 0, VOLUME_DIM - 1)
        end_idx = torch.clamp(z_center + radius + 1, 0, VOLUME_DIM - 1)

        z_mask[i, start_idx:end_idx] = 0

    z_pred_idx = torch.argmax(z_scores_full + z_mask, dim=1)

    # --- Stage 3: X-Refinement ---

    mri_slice_pred_z = mri_volume_batch[torch.arange(B), z_pred_idx, ...]

    # Find the 2 nearest angle *indices* for each item in the batch
    dists_x = torch.abs(mu_s1[:, 1].unsqueeze(1) - x_rots_all.unsqueeze(0))
    _, nearest_indices_x = dists_x.topk(2, dim=1, largest=False) # (B, 2)

    # Use the indices to gather the actual angle *values*
    x_angles_to_test = torch.gather(x_rots_all.expand(B, -1), 1, nearest_indices_x) # (B, 2)

    # Pass ONLY the (B, 2) tensor of angles to the model
    # x_scores_guided will have shape (B, 2)
    x_scores_guided = models['s3'](us_slice_batch, mri_slice_pred_z, angles_to_test=x_angles_to_test)

    # Find the best score *among the 2*
    best_idx_in_2_x = torch.argmax(x_scores_guided, dim=1) # (B,)

    # Gather the *original index* of the winner
    x_pred_idx = torch.gather(nearest_indices_x, 1, best_idx_in_2_x.unsqueeze(1)).squeeze(1)

    # --- Stage 4: Y-Refinement ---

    # We need the actual angle *value* to rotate the slice
    x_pred_angle = torch.gather(x_angles_to_test, 1, best_idx_in_2_x.unsqueeze(1)).squeeze(1)
    mri_slice_pred_zx = K.rotate(mri_slice_pred_z, x_pred_angle)

    # Find the 2 nearest angles for Y-rotation
    dists_y = torch.abs(mu_s1[:, 2].unsqueeze(1) - y_rots_all.unsqueeze(0))
    _, nearest_indices_y = dists_y.topk(2, dim=1, largest=False)
    y_angles_to_test = torch.gather(y_rots_all.expand(B, -1), 1, nearest_indices_y)

    # Pass the (B, 2) tensor of Y-angles to the model
    y_scores_guided = models['s4'](us_slice_batch, mri_slice_pred_zx, angles_to_test=y_angles_to_test)

    # Find the best score *among the 2*
    best_idx_in_2_y = torch.argmax(y_scores_guided, dim=1)

    # Gather the *original index* of the winner
    y_pred_idx = torch.gather(nearest_indices_y, 1, best_idx_in_2_y.unsqueeze(1)).squeeze(1)

    return z_pred_idx, x_pred_idx, y_pred_idx


def evaluate_full_pipeline_with_coarse_guidance(models, dataloader, confidence_multiplier=1.0):
    """
    Evaluates the end-to-end performance of the GUIDED pipeline (S1 -> S2 -> S3 -> S4).

    This function iterates through the test set, calls `get_guided_prediction` for each batch
    (which uses S1 guidance), and aggregates the final MAE and average inference time.

    Args:
        models (dict): Dictionary of trained models ('s1', 's2', 's3', 's4').
        dataloader (DataLoader): DataLoader providing the test dataset batches.
        confidence_multiplier (float): Multiplier for S1 sigma used in dynamic Z-windowing.
                                       Passed down to `get_guided_prediction`. Defaults to 1.0.

    Returns:
        tuple: (mae, preds_np, labels_np)
               - mae (np.ndarray): Array containing MAE for Z, X, Y.
               - preds_np (np.ndarray): NumPy array of all final predictions (N, 3).
               - labels_np (np.ndarray): NumPy array of all corresponding ground truth labels (N, 3).
    """

    # Set all models to evaluation mode
    for m in models.values(): m.eval()
    all_final_preds, all_true_labels = [], []
    total_time, num_samples = 0, 0

    with torch.no_grad():
        # Iterate through test batches
        for data in tqdm(dataloader, desc="Guided Pipeline Evaluation"):
            us_slice, mri_volume, label_s1, _, _, _ = [d.to(DEVICE) for d in data]
            B = us_slice.shape[0]

            start_time = time.perf_counter()

            # --- Call the core GUIDED prediction logic ---
            # This function handles the S1 guidance, dynamic Z window, and active S3/S4 guidance
            z_pred_idx, x_pred_idx, y_pred_idx = get_guided_prediction(
                models, us_slice, mri_volume, confidence_multiplier=confidence_multiplier
            )

            total_time += (time.perf_counter() - start_time)
            num_samples += B

            # --- Collate Results (same as unguided function) ---
            z_preds = z_pred_idx.cpu().numpy()
            x_preds = np.array([idx_to_x_rot[i.item()] for i in x_pred_idx.cpu()])
            y_preds = np.array([idx_to_y_rot[i.item()] for i in y_pred_idx.cpu()])

            all_final_preds.append(np.stack([z_preds, x_preds, y_preds], axis=1))
            all_true_labels.append(label_s1.cpu().numpy())

    # --- Aggregate Results (same as unguided function) ---
    preds_np = np.concatenate(all_final_preds)
    labels_np = np.concatenate(all_true_labels)
    # Calculate MAE
    mae = np.mean(np.abs(preds_np - labels_np), axis=0)
    # Calculate average time per slice (batched throughput)
    avg_time_ms = (total_time / num_samples) * 1000

    # Print summary results, indicating the dynamic window setting
    print("\n--- Guided Pipeline End-to-End Performance ---")
    print(f"Z-Slice search window: Dynamic (multiplier: {confidence_multiplier} * sigma)")
    print(f"MAE Z: {mae[0]:.3f} slices | MAE X-Rot: {mae[1]:.3f}° | MAE Y-Rot: {mae[2]:.3f}°")
    print(f"Average Inference Time: {avg_time_ms:.2f} ms per slice")

    # Return metrics and raw predictions/labels
    return mae, preds_np, labels_np

# --- 7. Plotting & Visualization ---

In [ ]:
# --- 7. Plotting & Visualization ---

def plot_training_history(histories):
    """
    Generates a grid of plots visualizing the training and validation performance
    metrics (Loss, MAE, Accuracy) across all four training stages.

    Creates a 4x3 grid where each row corresponds to a stage (S1, S2, S3, S4)
    and columns represent Loss, Validation MAE, and Validation Accuracy respectively.

    Args:
        histories (list): A list of dictionaries, where each dictionary contains the
                          training history ('train_loss', 'val_loss', 'val_mae', 'val_acc')
                          for one stage, in order (S1, S2, S3, S4).
    """

    """Plots loss, MAE, and accuracy curves for all training stages on a 4x3 grid."""

    # Set a visually appealing plot style
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, axes = plt.subplots(4, 3, figsize=(18, 20), dpi=100)
    fig.suptitle('Training Performance Across All Stages', fontsize=20, y=0.99)
    stage_titles = ["S1: Coupled Regression", "S2: Z-Refinement", "S3: X-Rot Refinement", "S4: Y-Rot Refinement"]

    # Iterate through the history data for each stage (S1 to S4)
    for i, (hist, title) in enumerate(zip(histories, stage_titles)):
        ax_loss, ax_mae, ax_acc = axes[i, 0], axes[i, 1], axes[i, 2]
        epochs = range(1, len(hist['train_loss']) + 1)

        # --- Column 1: Loss ---
        ax_loss.plot(epochs, hist['train_loss'], 'o-', label='Train Loss', color='b')
        ax_loss.plot(epochs, hist['val_loss'], 's--', label='Validation Loss', color='c')
        ax_loss.set_title(f"{title} - Loss")
        ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss'); ax_loss.legend()

        # --- Column 2: MAE ---
        if i == 0: # Stage 1 has 3 MAE values
            val_mae_z = [m[0] for m in hist['val_mae']]
            val_mae_x = [m[1] for m in hist['val_mae']]
            val_mae_y = [m[2] for m in hist['val_mae']]
            ax_mae.plot(epochs, val_mae_z, 'o-', label='Z MAE (slices)')
            ax_mae.plot(epochs, val_mae_x, 's-', label='X-Rot MAE (deg)')
            ax_mae.plot(epochs, val_mae_y, '^-', label='Y-Rot MAE (deg)')
        else: # Stages 2-4 have 1 MAE value
            unit = "slices" if i == 1 else "deg"
            ax_mae.plot(epochs, hist['val_mae'], 'o-', label=f'MAE ({unit})', color='r')
        ax_mae.set_title(f"{title} - Validation MAE")
        ax_mae.set_xlabel('Epoch'); ax_mae.set_ylabel('Mean Absolute Error'); ax_mae.legend()

        # --- Column 3: Accuracy ---
        if i == 0: # Stage 1 has 3 Accuracy values
            val_acc_z = [m[0] for m in hist['val_acc']]
            val_acc_x = [m[1] for m in hist['val_acc']]
            val_acc_y = [m[2] for m in hist['val_acc']]
            ax_acc.plot(epochs, val_acc_z, 'o-', label='Z Acc (tol<1)')
            ax_acc.plot(epochs, val_acc_x, 's-', label='X-Rot Acc (tol<2.5)')
            ax_acc.plot(epochs, val_acc_y, '^-', label='Y-Rot Acc (tol<2.5)')
        else: # Stages 2-4 have 1 Accuracy value
            ax_acc.plot(epochs, hist['val_acc'], 'o-', label='Accuracy', color='g')
        ax_acc.set_title(f"{title} - Validation Accuracy")
        ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy (%)'); ax_acc.legend(); ax_acc.set_ylim(0, 101)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()



def plot_evaluation_diagnostics(true_labels, pred_labels, rot_options, pipeline_name="Guided"):
    """
    Generates a comprehensive 3x3 dashboard visualizing the final evaluation results
    on the test set for either the guided or unguided pipeline.

    The dashboard includes:
    - Top Row: Histograms of absolute errors (Z, X, Y) with KDE curves.
    - Middle Row: Scatter plots showing Absolute Error vs. Ground Truth Misalignment for robustness analysis.
    - Bottom Row: Confusion matrices (Z is unbinned 50x50, X/Y are 5x5).
    - Overall Success Rate metrics are displayed in a text box.

    Args:
        true_labels (np.ndarray): NumPy array of ground truth labels (N, 3) for Z, X, Y.
        pred_labels (np.ndarray): NumPy array of corresponding predicted labels (N, 3).
        rot_options (iterable): List or array of the discrete rotation angle options (e.g., [0, 5, 10, 15, 20]).
        pipeline_name (str): Name of the pipeline ("Guided" or "Unguided") for the plot title.
                                Defaults to "Guided".
    """

    # Calculate absolute errors for each dimension
    errors = np.abs(true_labels - pred_labels)

    fig = plt.figure(figsize=(20, 18), dpi=110)
    gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.25)
    fig.suptitle(f'Comprehensive Diagnostics for {pipeline_name} Pipeline', fontsize=22, y=0.99)

    ax_hist_z = fig.add_subplot(gs[0, 0])
    ax_hist_x = fig.add_subplot(gs[0, 1])
    ax_hist_y = fig.add_subplot(gs[0, 2])
    ax_err_gt_z = fig.add_subplot(gs[1, 0])
    ax_err_gt_x = fig.add_subplot(gs[1, 1])
    ax_err_gt_y = fig.add_subplot(gs[1, 2])
    ax_cm_z = fig.add_subplot(gs[2, 0])
    ax_cm_x = fig.add_subplot(gs[2, 1])
    ax_cm_y = fig.add_subplot(gs[2, 2])

    # --- 1. Top Row: Plot Error Histograms ---
    sns.histplot(data=errors[:, 0], ax=ax_hist_z, kde=True, bins=max(1, int(errors[:, 0].max()) if errors[:, 0].size > 0 else 1))
    ax_hist_z.set_title('Z-Slice Error Distribution', fontsize=16)
    ax_hist_z.set_xlabel('Absolute Error (slices)')

    sns.histplot(data=errors[:, 1], ax=ax_hist_x, kde=True, color='C2', bins=len(rot_options))
    ax_hist_x.set_title('X-Rotation Error Distribution', fontsize=16)
    ax_hist_x.set_xlabel('Absolute Error (degrees)')

    sns.histplot(data=errors[:, 2], ax=ax_hist_y, kde=True, color='C3', bins=len(rot_options))
    ax_hist_y.set_title('Y-Rotation Error Distribution', fontsize=16)
    ax_hist_y.set_xlabel('Absolute Error (degrees)')

    # --- 2. Middle Row: Error vs. Ground Truth ---
    sns.scatterplot(x=true_labels[:, 0], y=errors[:, 0], ax=ax_err_gt_z, alpha=0.6)
    ax_err_gt_z.set_title('Robustness: Z-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_z.set_xlabel('Ground Truth Z-Slice')
    ax_err_gt_z.set_ylabel('Absolute Error (slices)')
    ax_err_gt_z.set_xlim(0,50)

    sns.scatterplot(x=true_labels[:, 1], y=errors[:, 1], ax=ax_err_gt_x, alpha=0.6, color='C2')
    ax_err_gt_x.set_title('Robustness: X-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_x.set_xlabel('Ground Truth X-Rotation (°)')
    ax_err_gt_x.set_ylabel('Absolute Error (degrees)')
    ax_err_gt_x.set_xlim(0,20)

    sns.scatterplot(x=true_labels[:, 2], y=errors[:, 2], ax=ax_err_gt_y, alpha=0.6, color='C3')
    ax_err_gt_y.set_title('Robustness: Y-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_y.set_xlabel('Ground Truth Y-Rotation (°)')
    ax_err_gt_y.set_ylabel('Absolute Error (degrees)')
    ax_err_gt_y.set_xlim(0,20)

    # --- 3. Bottom Row: Confusion Matrices ---

    z_labels = list(range(VOLUME_DIM))
    cm_z = confusion_matrix(true_labels[:, 0], pred_labels[:, 0], labels=z_labels)
    disp_z = ConfusionMatrixDisplay(confusion_matrix=cm_z, display_labels=z_labels)

    sns.heatmap(cm_z, ax=ax_cm_z, cmap='Blues', cbar=True, annot=False) # Annot=False for readability
    ax_cm_z.set_title('Z-Slice Confusion', fontsize=16)
    ax_cm_z.set_xlabel('Predicted Z-Slice')
    ax_cm_z.set_ylabel('True Z-Slice')

    tick_spacing = 5
    ax_cm_z.set_xticks(np.arange(0, VOLUME_DIM, tick_spacing) + 0.5)
    ax_cm_z.set_yticks(np.arange(0, VOLUME_DIM, tick_spacing) + 0.5)
    ax_cm_z.set_xticklabels(np.arange(0, VOLUME_DIM, tick_spacing))
    ax_cm_z.set_yticklabels(np.arange(0, VOLUME_DIM, tick_spacing))
    plt.setp(ax_cm_z.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    plt.setp(ax_cm_z.get_yticklabels(), rotation=0)

    cm_x = confusion_matrix(true_labels[:, 1], pred_labels[:, 1], labels=rot_options)
    disp_x = ConfusionMatrixDisplay(confusion_matrix=cm_x, display_labels=rot_options)
    disp_x.plot(ax=ax_cm_x, cmap='Greens', colorbar=False)
    ax_cm_x.set_title('X-Rotation Confusion', fontsize=16)

    cm_y = confusion_matrix(true_labels[:, 2], pred_labels[:, 2], labels=rot_options)
    disp_y = ConfusionMatrixDisplay(confusion_matrix=cm_y, display_labels=rot_options)
    disp_y.plot(ax=ax_cm_y, cmap='Reds', colorbar=False)
    ax_cm_y.set_title('Y-Rotation Confusion', fontsize=16)

    # --- 4. Success Rate ---
    z_tolerance = 1.0
    rot_tolerance = 2.5
    z_success = np.mean(errors[:, 0] <= z_tolerance) * 100
    x_success = np.mean(errors[:, 1] <= rot_tolerance) * 100
    y_success = np.mean(errors[:, 2] <= rot_tolerance) * 100

    success_text = (
        f"Success Rate (Z ≤ {z_tolerance:.1f} slice): {z_success:.1f}%\n"
        f"Success Rate (X ≤ {rot_tolerance:.1f}°): {x_success:.1f}%\n"
        f"Success Rate (Y ≤ {rot_tolerance:.1f}°): {y_success:.1f}%"
    )

    # Add the text box to the top-right corner of the figure
    fig.text(0.98, 0.98, success_text, transform=fig.transFigure,
             ha='right', va='top', fontsize=14,
             bbox=dict(boxstyle='round,pad=0.5', fc='aliceblue', ec='grey', lw=1, alpha=0.9))

    for ax in fig.get_axes():
        if not isinstance(ax.images, list) or len(ax.images) == 0:
             ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()



def visualize_predictions(models, test_dataset, full_dataset, original_mri_volumes, num_examples=5):
    """
    Visualizes a comparison between the input US slice, the network's predicted
    MRI slice alignment, and the ground truth MRI slice alignment for a few
    random examples selected from the pre-computed test set.

    Args:
        models (dict): Dictionary of trained models ('s1', 's2', 's3', 's4').
        test_dataset (Subset): The PyTorch Subset representing the test split
                               of the pre-computed data.
        full_dataset (PrecomputedTransformationDataset): The complete dataset object, needed to
                                                         map test set indices back to original data.
        original_mri_volumes (list): List of the original, untransformed 3D MRI volumes (NumPy).
        num_examples (int): The number of random examples to visualize. Defaults to 5.
    """

    if num_examples == 0:
        return

    plt.style.use('default')
    num_examples = min(num_examples, len(test_dataset))

    fig, axes = plt.subplots(num_examples, 3, figsize=(12, 4 * num_examples))
    if num_examples == 1:
        axes = axes.reshape(1, -1)

    fig.suptitle("Sample Predictions vs. Ground Truth", fontsize=20, y=0.99)

    # Set all models to evaluation mode
    for m in models.values(): m.eval()

    # Get random samples from the test dataset
    indices_to_show = np.random.choice(len(test_dataset), num_examples, replace=False)

    for i, data_idx in enumerate(indices_to_show):
        # Get the i-th sample from the test Subset
        us_slice_tensor, mri_volume_tensor, label_s1, _, _, _ = test_dataset[data_idx]

        # Find the original, untransformed MRI volume
        original_data_idx = test_dataset.indices[data_idx]
        _, mri_vol_idx, _ = full_dataset.us_precomputed_data[original_data_idx]
        base_mri_vol_np = original_mri_volumes[mri_vol_idx]

        # Add batch dimension and move to device
        us_slice_batch = us_slice_tensor.unsqueeze(0).to(DEVICE)
        mri_volume_batch = mri_volume_tensor.unsqueeze(0).to(DEVICE)

        # Run the full guided pipeline
        with torch.no_grad():
            z_pred_idx_tensor, x_pred_idx_tensor, y_pred_idx_tensor = get_guided_prediction(
                models, us_slice_batch, mri_volume_batch, confidence_multiplier=1.0
            )
            # .item() extracts the value from the 0-dim tensor
            z_pred_idx = z_pred_idx_tensor.item()
            x_pred_idx = x_pred_idx_tensor.item()
            y_pred_idx = y_pred_idx_tensor.item()

        # Final predicted parameters
        z_pred, x_rot_pred, y_rot_pred = z_pred_idx, idx_to_x_rot[x_pred_idx], idx_to_y_rot[y_pred_idx]

        # Ground truth parameters
        z_true, x_rot_true, y_rot_true = label_s1.numpy()

        # --- Plotting ---
        # Col 1: Input US Slice
        axes[i, 0].imshow(us_slice_tensor.squeeze(), cmap='gray')
        axes[i, 0].set_title("Input US Slice")
        axes[i, 0].axis('off')

        # Col 2: Predicted MRI Slice
        # Apply predicted rotations to the *original* MRI volume
        pred_rotated = scipy_rotate(base_mri_vol_np, angle=y_rot_pred, axes=(0, 2), reshape=False)
        pred_rotated = scipy_rotate(pred_rotated, angle=x_rot_pred, axes=(0, 1), reshape=False)
        axes[i, 1].imshow(pred_rotated[z_pred, :, :], cmap='gray')
        axes[i, 1].set_title(f"Predicted Match\nZ={z_pred}, X°={x_rot_pred}, Y°={y_rot_pred}")
        axes[i, 1].axis('off')

        # Col 3: Ground Truth MRI Slice
        # Apply true rotations to the *original* MRI volume
        true_rotated = scipy_rotate(base_mri_vol_np, angle=y_rot_true, axes=(0, 2), reshape=False)
        true_rotated = scipy_rotate(true_rotated, angle=x_rot_true, axes=(0, 1), reshape=False)
        axes[i, 2].imshow(true_rotated[int(round(z_true)), :, :], cmap='gray')
        axes[i, 2].set_title(f"Ground Truth Match\nZ={z_true:.1f}, X°={x_rot_true:.1f}, Y°={y_rot_true:.1f}")
        axes[i, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# --- 8. Main Execution ---

In [ ]:
# --- 8. Main Execution ---

# Pre-compute dataset to accelerate training
precomputed_samples = create_precomputed_dataset(us_vols, mri_vols)
full_dataset = PrecomputedTransformationDataset(precomputed_samples, mri_vols)

In [ ]:
# Split dataset
total_size = len(full_dataset)
train_size = int(TRAIN_SPLIT * total_size)
val_size = int(VAL_SPLIT * total_size)
test_size = total_size - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

print(f"\nDataset Split:")
print(f"Training set size:   {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size:       {len(test_dataset)}")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
# --- Model Training Pipeline ---
models_trained = {}
all_histories = []

# Stage 1: Coupled Regression
model_s1 = CoupledPredictor().to(DEVICE)
optimizer_s1 = optim.Adam(model_s1.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
criterion_nll = GaussianNLLLoss()
model_s1, history_s1 = run_training_stage(
    stage_num=1, stage_name="Coupled 3DOF Regression",
    model=model_s1, criterion_train=criterion_nll, criterion_val=criterion_nll,
    optimizer=optimizer_s1, num_epochs=NUM_EPOCHS_S1,
    train_loader=train_loader, val_loader=val_loader
)
models_trained['s1'] = model_s1
all_histories.append(history_s1)

In [ ]:
# Stage 2: Z-Slice Refinement
model_s2 = ZRefiner().to(DEVICE)
optimizer_s2 = optim.Adam(model_s2.parameters(), lr=LEARNING_RATE)
criterion_cls = CrossEntropyLoss()
model_s2, history_s2 = run_training_stage(
    stage_num=2, stage_name="Z-Slice Refinement",
    model=model_s2, criterion_train=criterion_cls, criterion_val=criterion_cls,
    optimizer=optimizer_s2, num_epochs=NUM_EPOCHS_S1,
    train_loader=train_loader, val_loader=val_loader,
    models_prev={}
)
models_trained['s2'] = model_s2
all_histories.append(history_s2)

In [ ]:
# Stage 3: X-Rotation Refinement
model_s3 = RotationRefiner(X_ROTATIONS).to(DEVICE)
optimizer_s3 = optim.Adam(model_s3.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
criterion_cls_smooth = CrossEntropyLoss(label_smoothing=0.2)
prev_models_s3 = {'s1': models_trained['s1'], 's2': models_trained['s2']}
model_s3, history_s3 = run_training_stage(
    stage_num=3, stage_name="X-Rotation Refinement (Hybrid Loss)", # Updated name for clarity
    model=model_s3, criterion_train=criterion_cls_smooth, criterion_val=criterion_cls,
    optimizer=optimizer_s3, num_epochs=NUM_EPOCHS_REFINEMENT,
    train_loader=train_loader, val_loader=val_loader,
    models_prev=prev_models_s3,
    use_hybrid_loss=True,
    lambda_ncc=0.5
)
models_trained['s3'] = model_s3
all_histories.append(history_s3)

In [ ]:
# Stage 4: Y-Rotation Refinement
model_s4 = RotationRefiner(Y_ROTATIONS).to(DEVICE)
optimizer_s4 = optim.Adam(model_s4.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
prev_models_s4 = {'s1': models_trained['s1'], 's2': models_trained['s2'], 's3': models_trained['s3']}
model_s4, history_s4 = run_training_stage(
    stage_num=4, stage_name="Y-Rotation Refinement (Hybrid Loss)",
    model=model_s4, criterion_train=criterion_cls_smooth, criterion_val=criterion_cls,
    optimizer=optimizer_s4, num_epochs=NUM_EPOCHS_REFINEMENT,
    train_loader=train_loader, val_loader=val_loader,
    models_prev=prev_models_s4,
    use_hybrid_loss=True,
    lambda_ncc=0.5
)
models_trained['s4'] = model_s4
all_histories.append(history_s4)

In [ ]:
# --- Final Evaluation and Visualization ---
print("\n" + "="*50)
print("      PERFORMING FINAL EVALUATION & VISUALIZATION ON TEST SET")
print("="*50)

# 1. Run the UNGUIDED pipeline
mae_unguided, preds_unguided, labels_unguided = evaluate_full_pipeline(
    models_trained, test_loader
)

# 2. Run the GUIDED pipeline
mae_guided, preds_guided, labels_guided = evaluate_full_pipeline_with_coarse_guidance(
    models_trained, test_loader, confidence_multiplier=1.0
)

In [ ]:
# 3. Plot training history
plot_training_history(all_histories)

In [ ]:
# 4. Plot diagnostics for the GUIDED pipeline
plot_evaluation_diagnostics(
    true_labels=labels_guided,
    pred_labels=preds_guided,
    rot_options=X_ROTATIONS
)

# Plot diagnostics for the UNGUIDED pipeline
plot_evaluation_diagnostics(
    true_labels=labels_unguided,
    pred_labels=preds_unguided,
    rot_options=X_ROTATIONS
)

In [ ]:
# 5. Visualize sample predictions
visualize_predictions(
    models=models_trained,
    test_dataset=test_dataset,
    full_dataset=full_dataset,
    original_mri_volumes=mri_vols,
    num_examples=5
)

print("\n" + "="*50)


# --- 9. Additional Plots and Tests ---

In [ ]:
# --- Printing Layers of the 2D CNN ---

# --- 1. Define a dictionary to store our layer outputs ---
activations = {}

def get_activation(name):
    """
    This is a "hook" function.
    It takes a name and returns another function that
    will be registered to a layer.
    """
    def hook(model, input, output):
        # We detach the output tensor from the computation graph
        # and store it in our dictionary.
        activations[name] = output.detach()
    return hook

# --- 2. Define a plotting function ---
def plot_activations(act_tensor, layer_name, num_channels_to_plot=16):
    """
    Plots the first 'num_channels_to_plot' feature maps
    from an activation tensor.
    """
    # Get the activations for the first image in the batch
    # and move to CPU
    act = act_tensor[0].cpu()

    # Ensure we don't try to plot more channels than exist
    num_channels_to_plot = min(num_channels_to_plot, act.shape[0])

    # Calculate grid size (e.g., 4x4 for 16 channels)
    grid_size = math.ceil(math.sqrt(num_channels_to_plot))

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(12, 12))
    fig.suptitle(f"Feature Map Activations - {layer_name}", fontsize=16)

    for i in range(num_channels_to_plot):
        ax = axes.flat[i]
        # Display the i-th channel
        ax.imshow(act[i], cmap='viridis')
        ax.set_title(f"Channel {i}")
        ax.axis('off')

    # Turn off unused subplots
    for i in range(num_channels_to_plot, grid_size * grid_size):
        axes.flat[i].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# --- 3. Register the hooks to your model ---
# We assume 'model_s1' is your trained Stage 1 model
model_s1.eval()

# Your slice_encoder is a Sequential module.
# Let's inspect its layers (they come from resnet.children()):
# slice_encoder[0] is conv1
# slice_encoder[1] is bn1
# slice_encoder[2] is relu
# slice_encoder[3] is maxpool
# slice_encoder[4] is layer1 (first residual block set)
# slice_encoder[5] is layer2
# slice_encoder[6] is layer3
# slice_encoder[7] is layer4 (last residual block set)

# Let's register hooks on an early, middle, and late layer
# (Similar to Layer 0, 2, and 5 in your example image)
model_s1.slice_encoder[0].register_forward_hook(get_activation('conv1'))
model_s1.slice_encoder[4].register_forward_hook(get_activation('layer1'))
model_s1.slice_encoder[6].register_forward_hook(get_activation('layer3'))

# --- 4. Run data through the model to trigger the hooks ---
# We just need one batch from the validation loader
with torch.no_grad():
    us_slice_batch, mri_batch, _, _, _, _ = next(iter(val_loader))
    us_slice_gpu = us_slice_batch.to(DEVICE)
    mri_volume_gpu = mri_batch.to(DEVICE)

    # Run the forward pass. This will trigger the hooks
    # and populate our 'activations' dictionary.
    _ = model_s1(us_slice_gpu, mri_volume_gpu)

# --- 5. Plot the results ---
# Now 'activations' dictionary is full. Let's plot them.

# Plot the output of the first conv layer
plot_activations(activations['conv1'], "ResNet-18 'conv1' (Early Layer)")

# Plot the output of the first residual block
plot_activations(activations['layer1'], "ResNet-18 'layer1' (Mid Layer)")

# Plot the output of the third residual block
plot_activations(activations['layer3'], "ResNet-18 'layer3' (Late Layer)")

print("Activation plots generated.")

# --- Guided vs Unguided Performance ---

In [ ]:
# --- Guided vs Unguided Performance ---

def compare_isolated_vs_connected_performance(models, dataloader, device):
    """
    Compares the performance of each model in two scenarios:
    1.  ISOLATED: Using ground-truth inputs (theoretical best performance).
    2.  CONNECTED: Using the predictions from the previous stage (real-world performance).

    This function explicitly quantifies the "Error Propagation Penalty".
    """
    print("\n" + "="*70)
    print("  COMPARING ISOLATED MODEL CAPACITY VS. CONNECTED PIPELINE PERFORMANCE")
    print("="*70)

    for m in models.values(): m.eval()

    # Lists to store errors for each scenario
    isolated_errors_z, isolated_errors_x, isolated_errors_y = [], [], []
    connected_errors_z, connected_errors_x, connected_errors_y = [], [], []

    with torch.no_grad():
        for data in tqdm(dataloader, desc="Comparing Pipeline Performance"):
            us_slice, mri_volume, label_s1, label_s2, label_s3, label_s4 = [d.to(device) for d in data]
            B = us_slice.shape[0]

            # --- 1. CONNECTED PIPELINE (Real-World) ---
            # Run the full guided pipeline where errors propagate
            z_pred_conn, x_pred_idx_conn, y_pred_idx_conn = get_guided_prediction(
                models, us_slice, mri_volume, confidence_multiplier=2.0
            )
            x_pred_conn = torch.tensor([idx_to_x_rot[i.item()] for i in x_pred_idx_conn], device=device)
            y_pred_conn = torch.tensor([idx_to_y_rot[i.item()] for i in y_pred_idx_conn], device=device)

            connected_errors_z.extend(torch.abs(z_pred_conn - label_s2).cpu().numpy())
            connected_errors_x.extend(torch.abs(x_pred_conn - label_s1[:, 1]).cpu().numpy())
            connected_errors_y.extend(torch.abs(y_pred_conn - label_s1[:, 2]).cpu().numpy())

            # --- 2. ISOLATED MODELS (Theoretical Best) ---
            # Test each stage by giving it perfect, ground-truth inputs

            # Stage 2 (Z-Refiner) in isolation
            z_scores_iso = models['s2'](us_slice, mri_volume)
            z_pred_iso = torch.argmax(z_scores_iso, dim=1)
            isolated_errors_z.extend(torch.abs(z_pred_iso - label_s2).cpu().numpy())

            # Stage 3 (X-Refiner) in isolation
            # Input: US slice + MRI slice at GROUND TRUTH Z
            mri_slice_gt_z = mri_volume[torch.arange(B), label_s2, ...]
            x_scores_iso = models['s3'](us_slice, mri_slice_gt_z)
            x_pred_idx_iso = torch.argmax(x_scores_iso, dim=1)
            x_pred_iso = torch.tensor([idx_to_x_rot[i.item()] for i in x_pred_idx_iso], device=device)
            isolated_errors_x.extend(torch.abs(x_pred_iso - label_s1[:, 1]).cpu().numpy())

            # Stage 4 (Y-Refiner) in isolation
            # Input: US slice + MRI slice at GT Z and GT X-rotation
            true_x_angles = label_s1[:, 1]
            mri_slice_gt_zx = K.rotate(mri_slice_gt_z, true_x_angles)
            y_scores_iso = models['s4'](us_slice, mri_slice_gt_zx)
            y_pred_idx_iso = torch.argmax(y_scores_iso, dim=1)
            y_pred_iso = torch.tensor([idx_to_y_rot[i.item()] for i in y_pred_idx_iso], device=device)
            isolated_errors_y.extend(torch.abs(y_pred_iso - label_s1[:, 2]).cpu().numpy())

    # --- 3. Calculate and Print Results ---
    mae_iso_z = np.mean(isolated_errors_z)
    mae_iso_x = np.mean(isolated_errors_x)
    mae_iso_y = np.mean(isolated_errors_y)

    mae_conn_z = np.mean(connected_errors_z)
    mae_conn_x = np.mean(connected_errors_x)
    mae_conn_y = np.mean(connected_errors_y)

    # Calculate the penalty
    penalty_z = mae_conn_z - mae_iso_z
    penalty_x = mae_conn_x - mae_iso_x
    penalty_y = mae_conn_y - mae_iso_y

    print("\n--- Pipeline Performance Analysis ---")
    print(f"{'Stage':<10} | {'Isolated MAE':<20} | {'Connected MAE':<20} | {'Error Propagation Penalty':<25}")
    print("-" * 80)
    print(f"{'S2 (Z)':<10} | {f'{mae_iso_z:.3f} slices':<20} | {f'{mae_conn_z:.3f} slices':<20} | {f'{penalty_z:+.3f} slices':<25}")
    print(f"{'S3 (X)':<10} | {f'{mae_iso_x:.3f}°':<20} | {f'{mae_conn_x:.3f}°':<20} | {f'{penalty_x:+.3f}°':<25}")
    print(f"{'S4 (Y)':<10} | {f'{mae_iso_y:.3f}°':<20} | {f'{mae_conn_y:.3f}°':<20} | {f'{penalty_y:+.3f}°':<25}")
    print("-" * 80)


compare_isolated_vs_connected_performance(
    models=models_trained,
    dataloader=test_loader,
    device=DEVICE
)


In [ ]:
# --- Seen vs Unseen Case Study ---

def _evaluate_single_instance(us_vol_np, mri_vol_np, models, device, ground_truth_params, idx_maps):
    """
    (Refactored Helper Function)
    Runs the pipeline for a single simulated misalignment and returns the results.
    This function is now silent and does not print.
    """
    z_true, x_rot_true, y_rot_true = ground_truth_params
    idx_to_x_rot, idx_to_y_rot = idx_maps['x'], idx_maps['y']

    # 1. Create the single "live" 2D US slice
    rotated_us = scipy_rotate(us_vol_np, angle=y_rot_true, axes=(0, 2), reshape=False)
    rotated_us = scipy_rotate(rotated_us, angle=x_rot_true, axes=(0, 1), reshape=False)
    us_slice_np = rotated_us[int(z_true), :, :].copy()

    # 2. Prepare tensors
    us_slice_tensor = torch.from_numpy(us_slice_np).float().unsqueeze(0).unsqueeze(0).to(device)
    mri_volume_tensor = torch.from_numpy(mri_vol_np).float().unsqueeze(1).unsqueeze(0).to(device)

    # 3. Run inference and time it
    with torch.no_grad():
        if device.type == 'cuda': torch.cuda.synchronize()
        start_time = time.perf_counter()

        z_pred_idx, x_pred_idx, y_pred_idx = get_guided_prediction(
            models, us_slice_tensor, mri_volume_tensor, confidence_multiplier=2.0
        )

        if device.type == 'cuda': torch.cuda.synchronize()
        end_time = time.perf_counter()

    inference_time_ms = (end_time - start_time) * 1000

    # 4. Convert predictions to real values
    z_pred = z_pred_idx.item()
    x_rot_pred = idx_to_x_rot[x_pred_idx.item()]
    y_rot_pred = idx_to_y_rot[y_pred_idx.item()]

    # 5. Calculate error
    error_z = abs(z_pred - z_true)
    error_x = abs(x_rot_pred - x_rot_true)
    error_y = abs(y_rot_pred - y_rot_true)

    return {
        "true": (z_true, x_rot_true, y_rot_true),
        "pred": (z_pred, x_rot_pred, y_rot_pred),
        "error": (error_z, error_x, error_y),
        "time_ms": inference_time_ms
    }



def plot_unseen_case_diagnostics(results, case_name, z_labels, x_labels, y_labels):
    """
    Plots a comprehensive 3x3 diagnostic dashboard for the unseen case evaluation.

    Includes:
    1. Error Histograms (with KDE) to show error distribution.
    2. Error vs. Ground Truth plots to assess robustness.
    3. Confusion Matrices for class-specific analysis (Z is UNBINNED).
    4. Success Rate metrics displayed directly on the figure.
    """
    # Unpack the results into NumPy arrays
    true_labels = np.array([r['true'] for r in results])
    pred_labels = np.array([r['pred'] for r in results])
    errors = np.array([r['error'] for r in results])

    # --- Create a 3x3 grid for the dashboard ---
    fig = plt.figure(figsize=(20, 18), dpi=110)
    gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.25)
    fig.suptitle(f'Comprehensive Diagnostics for Unseen Case: {case_name}', fontsize=22, y=0.99)

    # --- Create all 9 axes ---
    ax_hist_z = fig.add_subplot(gs[0, 0])
    ax_hist_x = fig.add_subplot(gs[0, 1])
    ax_hist_y = fig.add_subplot(gs[0, 2])
    ax_err_gt_z = fig.add_subplot(gs[1, 0])
    ax_err_gt_x = fig.add_subplot(gs[1, 1])
    ax_err_gt_y = fig.add_subplot(gs[1, 2])
    ax_cm_z = fig.add_subplot(gs[2, 0])
    ax_cm_x = fig.add_subplot(gs[2, 1])
    ax_cm_y = fig.add_subplot(gs[2, 2])

    # --- 1. Top Row: Error Histograms ---
    sns.histplot(data=errors[:, 0], ax=ax_hist_z, kde=True, bins=max(1, int(errors[:, 0].max()) if errors[:, 0].size > 0 else 1))
    ax_hist_z.set_title('Z-Slice Error Distribution', fontsize=16)
    ax_hist_z.set_xlabel('Absolute Error (slices)')

    sns.histplot(data=errors[:, 1], ax=ax_hist_x, kde=True, color='C2', bins=len(x_labels))
    ax_hist_x.set_title('X-Rotation Error Distribution', fontsize=16)
    ax_hist_x.set_xlabel('Absolute Error (degrees)')

    sns.histplot(data=errors[:, 2], ax=ax_hist_y, kde=True, color='C3', bins=len(y_labels))
    ax_hist_y.set_title('Y-Rotation Error Distribution', fontsize=16)
    ax_hist_y.set_xlabel('Absolute Error (degrees)')

    # --- 2. Middle Row: Error vs. Ground Truth Misalignment ---
    sns.scatterplot(x=true_labels[:, 0], y=errors[:, 0], ax=ax_err_gt_z, alpha=0.6)
    ax_err_gt_z.set_title('Robustness: Z-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_z.set_xlabel('Ground Truth Z-Slice')
    ax_err_gt_z.set_ylabel('Absolute Error (slices)')
    ax_err_gt_z.set_xlim(0,50)

    sns.scatterplot(x=true_labels[:, 1], y=errors[:, 1], ax=ax_err_gt_x, alpha=0.6, color='C2')
    ax_err_gt_x.set_title('Robustness: X-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_x.set_xlabel('Ground Truth X-Rotation (°)')
    ax_err_gt_x.set_ylabel('Absolute Error (degrees)')
    ax_err_gt_x.set_xlim(0,20)

    sns.scatterplot(x=true_labels[:, 2], y=errors[:, 2], ax=ax_err_gt_y, alpha=0.6, color='C3')
    ax_err_gt_y.set_title('Robustness: Y-Error vs. GT Misalignment', fontsize=16)
    ax_err_gt_y.set_xlabel('Ground Truth Y-Rotation (°)')
    ax_err_gt_y.set_ylabel('Absolute Error (degrees)')
    ax_err_gt_y.set_xlim(0,20)

    # --- 3. Bottom Row: Confusion Matrices ---

    # --- Z-Slice Confusion Matrix (UNBINNED) ---
    # Ensure z_labels is a list/array from 0 to VOLUME_DIM-1
    cm_z = confusion_matrix(true_labels[:, 0], pred_labels[:, 0], labels=z_labels)
    # Use seaborn heatmap for better control over labels
    sns.heatmap(cm_z, ax=ax_cm_z, cmap='Blues', cbar=True, annot=False) # Annot=False for readability
    ax_cm_z.set_title('Z-Slice Confusion', fontsize=16)
    ax_cm_z.set_xlabel('Predicted Z-Slice')
    ax_cm_z.set_ylabel('True Z-Slice')
    # Display fewer labels to avoid clutter
    tick_spacing = 5
    ax_cm_z.set_xticks(np.arange(0, VOLUME_DIM, tick_spacing) + 0.5)
    ax_cm_z.set_yticks(np.arange(0, VOLUME_DIM, tick_spacing) + 0.5)
    ax_cm_z.set_xticklabels(np.arange(0, VOLUME_DIM, tick_spacing))
    ax_cm_z.set_yticklabels(np.arange(0, VOLUME_DIM, tick_spacing))
    plt.setp(ax_cm_z.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    plt.setp(ax_cm_z.get_yticklabels(), rotation=0)

    # X-Rotation
    cm_x = confusion_matrix(true_labels[:, 1], pred_labels[:, 1], labels=x_labels)
    disp_x = ConfusionMatrixDisplay(confusion_matrix=cm_x, display_labels=x_labels)
    disp_x.plot(ax=ax_cm_x, cmap='Greens', colorbar=False)
    ax_cm_x.set_title('X-Rotation Confusion', fontsize=16)

    # Y-Rotation
    cm_y = confusion_matrix(true_labels[:, 2], pred_labels[:, 2], labels=y_labels)
    disp_y = ConfusionMatrixDisplay(confusion_matrix=cm_y, display_labels=y_labels)
    disp_y.plot(ax=ax_cm_y, cmap='Reds', colorbar=False)
    ax_cm_y.set_title('Y-Rotation Confusion', fontsize=16)

    # --- 4. Calculate and Display Success Rate ---
    z_tolerance = 1.0
    rot_tolerance = 2.5
    z_success = np.mean(errors[:, 0] <= z_tolerance) * 100
    x_success = np.mean(errors[:, 1] <= rot_tolerance) * 100
    y_success = np.mean(errors[:, 2] <= rot_tolerance) * 100

    success_text = (
        f"Success Rate (Z ≤ {z_tolerance:.1f} slice): {z_success:.1f}%\n"
        f"Success Rate (X ≤ {rot_tolerance:.1f}°): {x_success:.1f}%\n"
        f"Success Rate (Y ≤ {rot_tolerance:.1f}°): {y_success:.1f}%"
    )

    fig.text(0.98, 0.98, success_text, transform=fig.transFigure,
             ha='right', va='top', fontsize=14,
             bbox=dict(boxstyle='round,pad=0.5', fc='aliceblue', ec='grey', lw=1, alpha=0.9))

    # Add gridlines to relevant plots
    for ax in [ax_hist_z, ax_hist_x, ax_hist_y, ax_err_gt_z, ax_err_gt_x, ax_err_gt_y]:
        ax.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()



def run_full_unseen_case_evaluation(mri_path, us_path, models, device, num_variations=50):
    """
    Main controller to evaluate a single unseen case over multiple random misalignments.
    (This function remains mostly the same, just update the plot call)
    """
    print("=" * 60)
    case_name = os.path.basename(mri_path)
    print(f"STARTING FULL EVALUATION ON UNSEEN CASE: {case_name}")
    print(f"Number of random misalignments to test: {num_variations}")
    print("=" * 60)

    # 1. Load the data ONCE to be efficient
    mri_vol_np, us_vol_np = load_input_data(mri_path, us_path)
    if mri_vol_np is None:
        print("Failed to load data. Aborting evaluation.")
        return

    # Ensure models are in eval mode
    for m in models.values(): m.eval()

    # Package the index-to-rotation maps for cleaner passing
    idx_maps = {'x': idx_to_x_rot, 'y': idx_to_y_rot}

    all_results = []
    # 2. Loop to run multiple random simulations
    for _ in tqdm(range(num_variations), desc=f"Evaluating {case_name}"):
        # Generate a random set of ground truth parameters
        z_true = np.random.choice(Z_TRANSLATIONS)
        x_rot_true = np.random.choice(X_ROTATIONS)
        y_rot_true = np.random.choice(Y_ROTATIONS)
        gt_params = (z_true, x_rot_true, y_rot_true)

        # Run the pipeline for this single instance
        result = _evaluate_single_instance(us_vol_np, mri_vol_np, models, device, gt_params, idx_maps)
        all_results.append(result)

    # 3. Calculate and print aggregate statistics
    errors_np = np.array([r['error'] for r in all_results])
    times_np = np.array([r['time_ms'] for r in all_results])

    mae = np.mean(errors_np, axis=0)
    avg_time = np.mean(times_np)

    print("\n---" * 20)
    print(f"AGGREGATE RESULTS FOR UNSEEN CASE: {case_name}")
    print(f"Mean Absolute Error (Z):   {mae[0]:.3f} slices")
    print(f"Mean Absolute Error (X-Rot): {mae[1]:.3f}°")
    print(f"Mean Absolute Error (Y-Rot): {mae[2]:.3f}°")
    print(f"Average Inference Time:    {avg_time:.2f} ms per slice")
    print("---" * 20)

    # 4. Generate diagnostic plots
    plot_unseen_case_diagnostics(all_results, case_name, Z_TRANSLATIONS, X_ROTATIONS, Y_ROTATIONS)

new_mri_path_unseen = "INPUT/Case1-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case1-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
new_mri_path_unseen = "INPUT/Case2-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case2-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
new_mri_path_unseen = "INPUT/Case3-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case3-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
new_mri_path_unseen = "INPUT/Case4-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case4-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
new_mri_path_unseen = "INPUT/Case5-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case5-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
new_mri_path_unseen = "INPUT/Case6-FLAIR.nii.gz"
new_us_path_unseen = "INPUT/Case6-US-during.nii.gz"

run_full_unseen_case_evaluation(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_variations=100  # You can change this number
)

In [ ]:
def visualize_unseen_case_predictions(mri_path, us_path, models, device, num_examples=5):
    """
    Loads an unseen US/MRI pair, simulates random misalignments,
    runs the prediction pipeline, and visualizes the results.
    """
    print("-" * 50)
    case_name = os.path.basename(mri_path)
    print(f"Visualizing predictions for unseen case: {case_name}")

    # 1. Load the unseen 3D volumes
    mri_vol_np, us_vol_np = load_input_data(mri_path, us_path)
    if mri_vol_np is None:
        print("Failed to load data. Aborting visualization.")
        return

    # Set models to eval mode
    for m in models.values(): m.eval()

    # --- Setup Plotting ---
    if num_examples == 0: return
    plt.style.use('default')
    fig, axes = plt.subplots(num_examples, 3, figsize=(12, 4 * num_examples))
    if num_examples == 1: axes = axes.reshape(1, -1)
    fig.suptitle(f"Sample Predictions vs. Ground Truth for Unseen Case: {case_name}", fontsize=16, y=0.99)

    # --- Loop for specified number of examples ---
    for i in range(num_examples):
        # 2. Generate random ground truth misalignment for this example
        z_true = np.random.choice(Z_TRANSLATIONS)
        x_rot_true = np.random.choice(X_ROTATIONS)
        y_rot_true = np.random.choice(Y_ROTATIONS)
        ground_truth_params = (z_true, x_rot_true, y_rot_true)

        # 3. Create the simulated "live" 2D US slice
        rotated_us = scipy_rotate(us_vol_np, angle=y_rot_true, axes=(0, 2), reshape=False)
        rotated_us = scipy_rotate(rotated_us, angle=x_rot_true, axes=(0, 1), reshape=False)
        us_slice_np = rotated_us[int(z_true), :, :].copy()

        # 4. Prepare tensors (Batch size = 1)
        us_slice_tensor = torch.from_numpy(us_slice_np).float().unsqueeze(0).unsqueeze(0).to(device)
        mri_volume_tensor = torch.from_numpy(mri_vol_np).float().unsqueeze(1).unsqueeze(0).to(device)

        # 5. Run the full guided pipeline
        with torch.no_grad():
            z_pred_idx, x_pred_idx, y_pred_idx = get_guided_prediction(
                models, us_slice_tensor, mri_volume_tensor, confidence_multiplier=2.0
            )
            z_pred = z_pred_idx.item()
            x_rot_pred = idx_to_x_rot[x_pred_idx.item()]
            y_rot_pred = idx_to_y_rot[y_pred_idx.item()]

        # --- 6. Plotting ---
        # Col 1: Input US Slice (The simulated "live" slice)
        axes[i, 0].imshow(us_slice_np, cmap='gray')
        axes[i, 0].set_title("Input US Slice (Simulated)")
        axes[i, 0].axis('off')

        # Col 2: Predicted MRI Slice
        # Apply predicted rotations to the ORIGINAL UNSEEN MRI volume
        pred_rotated_mri = scipy_rotate(mri_vol_np, angle=y_rot_pred, axes=(0, 2), reshape=False)
        pred_rotated_mri = scipy_rotate(pred_rotated_mri, angle=x_rot_pred, axes=(0, 1), reshape=False)
        axes[i, 1].imshow(pred_rotated_mri[z_pred, :, :], cmap='gray')
        axes[i, 1].set_title(f"Predicted Match\nZ={z_pred}, X°={x_rot_pred:.0f}, Y°={y_rot_pred:.0f}")
        axes[i, 1].axis('off')

        # Col 3: Ground Truth MRI Slice
        # Apply TRUE rotations to the ORIGINAL UNSEEN MRI volume
        true_rotated_mri = scipy_rotate(mri_vol_np, angle=y_rot_true, axes=(0, 2), reshape=False)
        true_rotated_mri = scipy_rotate(true_rotated_mri, angle=x_rot_true, axes=(0, 1), reshape=False)
        axes[i, 2].imshow(true_rotated_mri[int(round(z_true)), :, :], cmap='gray')
        axes[i, 2].set_title(f"Ground Truth Match\nZ={z_true:.0f}, X°={x_rot_true:.0f}, Y°={y_rot_true:.0f}")
        axes[i, 2].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


new_mri_path_unseen = "INPUT/Case1-FLAIR.nii.gz" # Your unseen case
new_us_path_unseen = "INPUT/Case1-US-during.nii.gz"

visualize_unseen_case_predictions(
    mri_path=new_mri_path_unseen,
    us_path=new_us_path_unseen,
    models=models_trained,
    device=DEVICE,
    num_examples=5 # Show 5 random misalignments
)
